# openoppsdb manager

This notebook is connected to `wyattowalsh/openoppsdb`. Schedule it with a daily Kaggle cron cadence such as `0 6 * * *`. Each run installs OpenOpps from GitHub, copies the newest `/kaggle/input/**/openoppsdb.sqlite` snapshot into `/kaggle/working/openoppsdb/openoppsdb.sqlite`, runs `openopps sync --metrics-json`, captures status and coverage evidence for the private quality gate, prepares SQLite/CSV/Parquet artifacts, writes in-database table and column metadata, prunes private evidence from the upload directory, and deploys a new dataset version only after the quality gate passes.


In [ ]:
#@title Initialize
from __future__ import annotations

import base64
import csv
import hashlib
from functools import lru_cache
from html import unescape
import json
import os
from pathlib import Path
import re
import shutil
import sqlite3
import subprocess
import sys
import time
from datetime import UTC, datetime
import urllib.request

DATASET_ID = os.environ.get(
    "OPENOPPS_KAGGLE_DATASET",
    "wyattowalsh/openoppsdb",
)
PACKAGE_SPEC = os.environ.get(
    "OPENOPPS_PACKAGE_SPEC",
    "git+https://github.com/wyattowalsh/openopps.git@main",
)
OUTPUT_DIR = Path(
    os.environ.get(
        "OPENOPPS_KAGGLE_OUTPUT_DIR",
        "/kaggle/working/openoppsdb",
    )
)
DB_PATH = OUTPUT_DIR / "openoppsdb.sqlite"
GENERATOR_SCRIPT = OUTPUT_DIR / "generate_kaggle_metadata.py"
CSV_DIR = "exports/csv"
PARQUET_DIR = "exports/parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
INPUT_DB_GLOB = "**/openoppsdb.sqlite"
INPUT_JOB_VERSIONS_PARQUET_GLOB = "**/exports/parquet/job_versions.parquet"
INPUT_JOB_PAYLOAD_SNAPSHOTS_PARQUET_GLOB = "**/exports/parquet/job_payload_snapshots.parquet"
GENERATOR_SCRIPT_URL = os.environ.get(
    "OPENOPPS_GENERATOR_SCRIPT_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/scripts/generate_kaggle_metadata.py",
)
DATASET_IMAGE_URL = os.environ.get(
    "OPENOPPS_DATASET_IMAGE_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/docs/public/social/openoppsdb.png",
)
OPENOPPS_SYNC_ENV_DEFAULTS = {
    "OPENOPPS_BOARD_CONCURRENCY": "80",
    "OPENOPPS_HTTP_TIMEOUT": "20",
    "OPENOPPS_JOB_ROUTE_FRESHNESS_SECONDS": "86400",
    "OPENOPPS_JOB_ROUTE_TIMEOUT_SECONDS": "180",
    "OPENOPPS_MAX_CONNECTIONS": "120",
    "OPENOPPS_PROVIDER_CONCURRENCY": "80",
    "OPENOPPS_RETRY_ATTEMPTS": "2",
    "OPENOPPS_SOURCE_CONCURRENCY": "40",
    "OPENOPPS_SOURCE_FRESHNESS_SECONDS": "86400",
    "OPENOPPS_SOURCE_TIMEOUT_SECONDS": "120"
}
SKILL_CATALOG = [
    [
        "Programming Languages",
        [
            [
                "Python",
                [
                    "python"
                ]
            ],
            [
                "JavaScript",
                [
                    "javascript",
                    "java script",
                    "js"
                ]
            ],
            [
                "TypeScript",
                [
                    "typescript",
                    "type script",
                    "ts"
                ]
            ],
            [
                "Java",
                [
                    "java"
                ]
            ],
            [
                "C++",
                [
                    "c++",
                    "cplusplus"
                ]
            ],
            [
                "C#",
                [
                    "c#",
                    "csharp"
                ]
            ],
            [
                "Ruby",
                [
                    "ruby"
                ]
            ],
            [
                "PHP",
                [
                    "php"
                ]
            ],
            [
                "Swift",
                [
                    "swift"
                ]
            ],
            [
                "Kotlin",
                [
                    "kotlin"
                ]
            ],
            [
                "Rust",
                [
                    "rust"
                ]
            ],
            [
                "Scala",
                [
                    "scala"
                ]
            ],
            [
                "Golang",
                [
                    "golang"
                ]
            ],
            [
                "HTML",
                [
                    "html"
                ]
            ],
            [
                "CSS",
                [
                    "css"
                ]
            ]
        ]
    ],
    [
        "Frontend",
        [
            [
                "React",
                [
                    "react",
                    "reactjs",
                    "react.js"
                ]
            ],
            [
                "React Native",
                [
                    "react native"
                ]
            ],
            [
                "Angular",
                [
                    "angular"
                ]
            ],
            [
                "Vue",
                [
                    "vue",
                    "vuejs",
                    "vue.js"
                ]
            ],
            [
                "Svelte",
                [
                    "svelte"
                ]
            ],
            [
                "Next.js",
                [
                    "next.js",
                    "nextjs"
                ]
            ],
            [
                "Tailwind CSS",
                [
                    "tailwind",
                    "tailwind css"
                ]
            ],
            [
                "Web Components",
                [
                    "web components"
                ]
            ]
        ]
    ],
    [
        "Backend",
        [
            [
                "Node.js",
                [
                    "node.js",
                    "nodejs"
                ]
            ],
            [
                "Django",
                [
                    "django"
                ]
            ],
            [
                "Flask",
                [
                    "flask"
                ]
            ],
            [
                "FastAPI",
                [
                    "fastapi",
                    "fast api"
                ]
            ],
            [
                "Ruby on Rails",
                [
                    "ruby on rails",
                    "rails"
                ]
            ],
            [
                "Spring",
                [
                    "spring boot",
                    "spring framework"
                ]
            ],
            [
                "GraphQL",
                [
                    "graphql"
                ]
            ],
            [
                "REST APIs",
                [
                    "rest api",
                    "rest apis",
                    "restful api",
                    "restful apis"
                ]
            ],
            [
                "Microservices",
                [
                    "microservices",
                    "micro-services"
                ]
            ]
        ]
    ],
    [
        "Data and AI",
        [
            [
                "Machine Learning",
                [
                    "machine learning"
                ]
            ],
            [
                "Deep Learning",
                [
                    "deep learning"
                ]
            ],
            [
                "Generative AI",
                [
                    "generative ai",
                    "genai",
                    "gen ai"
                ]
            ],
            [
                "LLM",
                [
                    "llm",
                    "large language model",
                    "large language models"
                ]
            ],
            [
                "NLP",
                [
                    "nlp",
                    "natural language processing"
                ]
            ],
            [
                "Computer Vision",
                [
                    "computer vision"
                ]
            ],
            [
                "PyTorch",
                [
                    "pytorch"
                ]
            ],
            [
                "TensorFlow",
                [
                    "tensorflow"
                ]
            ],
            [
                "scikit-learn",
                [
                    "scikit-learn",
                    "sklearn"
                ]
            ],
            [
                "Pandas",
                [
                    "pandas"
                ]
            ],
            [
                "NumPy",
                [
                    "numpy"
                ]
            ],
            [
                "Spark",
                [
                    "apache spark",
                    "spark"
                ]
            ],
            [
                "Airflow",
                [
                    "airflow",
                    "apache airflow"
                ]
            ],
            [
                "dbt",
                [
                    "dbt"
                ]
            ],
            [
                "Analytics",
                [
                    "analytics"
                ]
            ],
            [
                "Experimentation",
                [
                    "experimentation",
                    "a/b testing",
                    "ab testing"
                ]
            ]
        ]
    ],
    [
        "Cloud and Infrastructure",
        [
            [
                "AWS",
                [
                    "aws",
                    "amazon web services"
                ]
            ],
            [
                "Azure",
                [
                    "azure",
                    "microsoft azure"
                ]
            ],
            [
                "Google Cloud",
                [
                    "google cloud",
                    "gcp"
                ]
            ],
            [
                "Kubernetes",
                [
                    "kubernetes",
                    "k8s"
                ]
            ],
            [
                "Docker",
                [
                    "docker"
                ]
            ],
            [
                "Terraform",
                [
                    "terraform"
                ]
            ],
            [
                "Helm",
                [
                    "helm"
                ]
            ],
            [
                "Linux",
                [
                    "linux"
                ]
            ],
            [
                "DevOps",
                [
                    "devops"
                ]
            ],
            [
                "SRE",
                [
                    "sre",
                    "site reliability"
                ]
            ],
            [
                "CI/CD",
                [
                    "ci/cd",
                    "cicd",
                    "continuous integration"
                ]
            ],
            [
                "GitHub Actions",
                [
                    "github actions"
                ]
            ],
            [
                "Jenkins",
                [
                    "jenkins"
                ]
            ],
            [
                "Observability",
                [
                    "observability"
                ]
            ],
            [
                "Prometheus",
                [
                    "prometheus"
                ]
            ],
            [
                "Grafana",
                [
                    "grafana"
                ]
            ]
        ]
    ],
    [
        "Databases",
        [
            [
                "SQL",
                [
                    "sql"
                ]
            ],
            [
                "PostgreSQL",
                [
                    "postgresql",
                    "postgres"
                ]
            ],
            [
                "MySQL",
                [
                    "mysql"
                ]
            ],
            [
                "SQLite",
                [
                    "sqlite"
                ]
            ],
            [
                "MongoDB",
                [
                    "mongodb",
                    "mongo"
                ]
            ],
            [
                "Redis",
                [
                    "redis"
                ]
            ],
            [
                "Elasticsearch",
                [
                    "elasticsearch",
                    "elastic search"
                ]
            ],
            [
                "Kafka",
                [
                    "kafka",
                    "apache kafka"
                ]
            ],
            [
                "DynamoDB",
                [
                    "dynamodb",
                    "dynamo db"
                ]
            ],
            [
                "Snowflake",
                [
                    "snowflake"
                ]
            ],
            [
                "BigQuery",
                [
                    "bigquery",
                    "big query"
                ]
            ],
            [
                "Databricks",
                [
                    "databricks"
                ]
            ]
        ]
    ],
    [
        "Security and Compliance",
        [
            [
                "Security",
                [
                    "security",
                    "cybersecurity",
                    "cyber security"
                ]
            ],
            [
                "SOC 2",
                [
                    "soc 2",
                    "soc2"
                ]
            ],
            [
                "HIPAA",
                [
                    "hipaa"
                ]
            ],
            [
                "GDPR",
                [
                    "gdpr"
                ]
            ],
            [
                "IAM",
                [
                    "iam",
                    "identity and access management"
                ]
            ],
            [
                "OAuth",
                [
                    "oauth",
                    "oauth2"
                ]
            ],
            [
                "SAML",
                [
                    "saml"
                ]
            ],
            [
                "Incident Response",
                [
                    "incident response"
                ]
            ],
            [
                "Vulnerability Management",
                [
                    "vulnerability management"
                ]
            ],
            [
                "Penetration Testing",
                [
                    "penetration testing",
                    "pentesting"
                ]
            ]
        ]
    ],
    [
        "Product and Design",
        [
            [
                "Product Management",
                [
                    "product management",
                    "product manager"
                ]
            ],
            [
                "Roadmapping",
                [
                    "roadmap",
                    "roadmapping"
                ]
            ],
            [
                "User Research",
                [
                    "user research",
                    "ux research"
                ]
            ],
            [
                "UX",
                [
                    "ux",
                    "user experience"
                ]
            ],
            [
                "UI",
                [
                    "ui",
                    "user interface"
                ]
            ],
            [
                "Figma",
                [
                    "figma"
                ]
            ],
            [
                "Design Systems",
                [
                    "design system",
                    "design systems"
                ]
            ],
            [
                "Prototyping",
                [
                    "prototype",
                    "prototyping"
                ]
            ],
            [
                "Growth",
                [
                    "growth"
                ]
            ]
        ]
    ],
    [
        "GTM and Customer",
        [
            [
                "Sales",
                [
                    "sales"
                ]
            ],
            [
                "Marketing",
                [
                    "marketing"
                ]
            ],
            [
                "Account Executive",
                [
                    "account executive"
                ]
            ],
            [
                "Customer Success",
                [
                    "customer success"
                ]
            ],
            [
                "CRM",
                [
                    "crm"
                ]
            ],
            [
                "Salesforce",
                [
                    "salesforce"
                ]
            ],
            [
                "HubSpot",
                [
                    "hubspot"
                ]
            ],
            [
                "Demand Generation",
                [
                    "demand generation"
                ]
            ],
            [
                "Partnerships",
                [
                    "partnerships"
                ]
            ],
            [
                "Support",
                [
                    "customer support",
                    "technical support"
                ]
            ]
        ]
    ],
    [
        "Operations and Finance",
        [
            [
                "Finance",
                [
                    "finance"
                ]
            ],
            [
                "Accounting",
                [
                    "accounting"
                ]
            ],
            [
                "FP&A",
                [
                    "fp&a",
                    "fpa"
                ]
            ],
            [
                "Payroll",
                [
                    "payroll"
                ]
            ],
            [
                "Recruiting",
                [
                    "recruiting",
                    "talent acquisition"
                ]
            ],
            [
                "People Operations",
                [
                    "people operations",
                    "people ops"
                ]
            ],
            [
                "Legal",
                [
                    "legal"
                ]
            ],
            [
                "Procurement",
                [
                    "procurement"
                ]
            ],
            [
                "Supply Chain",
                [
                    "supply chain"
                ]
            ],
            [
                "RevOps",
                [
                    "revops",
                    "revenue operations"
                ]
            ]
        ]
    ],
    [
        "Healthcare and Science",
        [
            [
                "Clinical",
                [
                    "clinical"
                ]
            ],
            [
                "Healthcare",
                [
                    "healthcare",
                    "health care"
                ]
            ],
            [
                "Biotech",
                [
                    "biotech",
                    "biotechnology"
                ]
            ],
            [
                "Pharma",
                [
                    "pharma",
                    "pharmaceutical"
                ]
            ],
            [
                "FDA",
                [
                    "fda"
                ]
            ],
            [
                "Laboratory",
                [
                    "laboratory",
                    "lab operations"
                ]
            ],
            [
                "Genomics",
                [
                    "genomics"
                ]
            ]
        ]
    ]
]
DATASET_METADATA = {'description': '# OpenOppsDB\n'
                '\n'
                'OpenOppsDB is a versioned public hiring-board ledger generated by the OpenOpps '
                'CLI. It tracks discovered company boards, executable provider routes, normalized '
                'job identities, versioned job content, raw provider payload snapshots, and sync '
                'observations over time.\n'
                '\n'
                '## What is included\n'
                '\n'
                '- `openoppsdb.sqlite`: the relational SQLite ledger, including metadata tables '
                'named `openopps_tables` and `openopps_columns`. To keep Kaggle table previews '
                'indexable, this SQLite copy nulls bulky text/JSON mirrors after export: '
                '`job_versions.description_html`, `job_versions.job_description`, and '
                '`job_payload_snapshots.payload`.\n'
                '- `exports/csv/*.csv`: full table exports for spreadsheet and lightweight '
                'analysis workflows, including rendered HTML descriptions, structured '
                'job-description JSON, and raw payloads.\n'
                '- `exports/parquet/*.parquet`: full table exports for Python, DuckDB, Polars, '
                'Spark, and warehouse workflows, including rendered HTML descriptions, structured '
                'job-description JSON, and raw payloads.\n'
                '\n'
                '## How updates work\n'
                '\n'
                'The connected Kaggle notebook `openoppsdb-manager` is intended to run once per '
                'day on a Kaggle cron schedule. Each run installs OpenOpps from GitHub, copies the '
                'current `openoppsdb.sqlite` from this dataset, runs `openopps sync '
                '--metrics-json`, captures private run evidence for the quality gate, exports '
                'every SQLite table to CSV and Parquet, regenerates Kaggle field metadata, prunes '
                'private manager evidence from the upload directory, and publishes a new dataset '
                'version only when the quality gate passes. The public file surface is '
                'intentionally limited to `openoppsdb.sqlite`, `exports/csv/*.csv`, and '
                '`exports/parquet/*.parquet`.\n'
                '\n'
                '## Quick start\n'
                '\n'
                '```python\n'
                'import sqlite3\n'
                'import polars as pl\n'
                '\n'
                "conn = sqlite3.connect('/kaggle/input/openoppsdb/openoppsdb.sqlite')\n"
                "jobs = pl.read_database('select * from jobs limit 10', conn)\n"
                'versions = '
                "pl.read_parquet('/kaggle/input/openoppsdb/exports/parquet/job_versions.parquet')\n"
                '```\n'
                '\n'
                '## Notes and limitations\n'
                '\n'
                'OpenOpps only uses public endpoints and public pages. Provider payloads are '
                'preserved for auditability, but normalized fields should be treated as '
                'best-effort public-data extraction rather than official ATS records. A row can '
                'appear across multiple source catalogs; durable keys and sync observation tables '
                'are provided so downstream users can reason about provenance and change '
                'history.\n',
 'expectedUpdateFrequency': 'daily',
 'id': 'wyattowalsh/openoppsdb',
 'image': 'dataset-cover-image.png',
 'isPrivate': False,
 'keywords': ['business', 'internet', 'software', 'tabular'],
 'licenses': [{'name': 'CC0-1.0'}],
 'resources': [{'description': 'Full SQLite ledger with source, board, provider route, job '
                               'lifecycle, version history, raw payload snapshot, sync '
                               'observation, and in-DB table and column metadata tables. The '
                               'SQLite upload nulls bulky text and JSON mirrors so Kaggle can '
                               'index table previews; full rendered descriptions, structured '
                               'job-description JSON, and raw payloads remain in the CSV and '
                               'Parquet exports.',
                'name': 'openopps_database',
                'path': 'openoppsdb.sqlite',
                'tables': [{'description': 'Durable source catalogs that discover company boards.',
                            'name': 'sources',
                            'schema': {'fields': [{'description': 'Stable local source key.',
                                                   'name': 'key',
                                                   'type': 'string'},
                                                  {'description': 'Canonical source URL or '
                                                                  'synthetic manual source URI.',
                                                   'name': 'url',
                                                   'type': 'string'},
                                                  {'description': 'Source adapter identifier.',
                                                   'name': 'provider_id',
                                                   'type': 'id'},
                                                  {'description': 'Whether unscoped syncs include '
                                                                  'this source.',
                                                   'name': 'enabled',
                                                   'type': 'boolean'},
                                                  {'description': 'Provider version metadata.',
                                                   'name': 'version',
                                                   'type': 'string'},
                                                  {'description': 'Source configuration and sync '
                                                                  'metadata.',
                                                   'name': 'raw_metadata',
                                                   'type': 'string'},
                                                  {'description': 'Unknown top-level record fields '
                                                                  'preserved across storage round '
                                                                  'trips.',
                                                   'name': 'extra_payload',
                                                   'type': 'string'},
                                                  {'description': 'Last successful source sync '
                                                                  'timestamp.',
                                                   'name': 'synced_at',
                                                   'type': 'string'}]},
                            'title': 'Sources'},
                           {'description': 'Durable normalized company or organization hiring '
                                           'boards.',
                            'name': 'boards',
                            'schema': {'fields': [{'description': 'Stable normalized board key.',
                                                   'name': 'key',
                                                   'type': 'string'},
                                                  {'description': 'Source key that emitted this '
                                                                  'board.',
                                                   'name': 'source_key',
                                                   'type': 'id'},
                                                  {'description': 'All sources that currently '
                                                                  'contain this board domain.',
                                                   'name': 'source_keys',
                                                   'type': 'string'},
                                                  {'description': 'Source-specific emitted board '
                                                                  'keys merged into this board.',
                                                   'name': 'source_board_keys',
                                                   'type': 'string'},
                                                  {'description': 'Provider-native board '
                                                                  'identifier.',
                                                   'name': 'remote_id',
                                                   'type': 'id'},
                                                  {'description': 'Provider-native slug.',
                                                   'name': 'remote_slug',
                                                   'type': 'string'},
                                                  {'description': 'Company or board display name.',
                                                   'name': 'name',
                                                   'type': 'string'},
                                                  {'description': 'Normalized website domain.',
                                                   'name': 'domain',
                                                   'type': 'string'},
                                                  {'description': 'Canonical company website URL.',
                                                   'name': 'website_url',
                                                   'type': 'url'},
                                                  {'description': 'Provider-supplied board '
                                                                  'description.',
                                                   'name': 'description',
                                                   'type': 'string'},
                                                  {'description': 'Industry or market tags.',
                                                   'name': 'markets',
                                                   'type': 'string'},
                                                  {'description': 'Office or hiring locations.',
                                                   'name': 'locations',
                                                   'type': 'string'},
                                                  {'description': 'Employee or team-size estimate.',
                                                   'name': 'staff_count',
                                                   'type': 'integer'},
                                                  {'description': 'Approximate number of open '
                                                                  'jobs.',
                                                   'name': 'num_jobs_hint',
                                                   'type': 'integer'},
                                                  {'description': 'Unmodified upstream board '
                                                                  'payload.',
                                                   'name': 'raw_payload',
                                                   'type': 'string'},
                                                  {'description': 'Unknown top-level record fields '
                                                                  'preserved across storage round '
                                                                  'trips.',
                                                   'name': 'extra_payload',
                                                   'type': 'string'},
                                                  {'description': 'Last successful board sync '
                                                                  'timestamp.',
                                                   'name': 'synced_at',
                                                   'type': 'string'}]},
                            'title': 'Boards'},
                           {'description': 'Durable provider routes that connect boards to '
                                           'upstream systems.',
                            'name': 'board_providers',
                            'schema': {'fields': [{'description': 'Stable route primary key.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Source key that reported this '
                                                                  'route.',
                                                   'name': 'source_key',
                                                   'type': 'id'},
                                                  {'description': 'Board key this route belongs '
                                                                  'to.',
                                                   'name': 'board_key',
                                                   'type': 'id'},
                                                  {'description': 'Provider adapter identifier.',
                                                   'name': 'provider_id',
                                                   'type': 'id'},
                                                  {'description': 'Human-readable upstream route '
                                                                  'label.',
                                                   'name': 'label',
                                                   'type': 'string'},
                                                  {'description': 'Normalized provider support '
                                                                  'level.',
                                                   'name': 'support_level',
                                                   'type': 'string'},
                                                  {'description': 'Approximate provider-reported '
                                                                  'job count.',
                                                   'name': 'count_hint',
                                                   'type': 'integer'},
                                                  {'description': 'Hosted job board URL.',
                                                   'name': 'board_url',
                                                   'type': 'url'},
                                                  {'description': 'Provider-specific board token '
                                                                  'or slug.',
                                                   'name': 'token',
                                                   'type': 'string'},
                                                  {'description': 'Multi-tenant provider host.',
                                                   'name': 'host',
                                                   'type': 'string'},
                                                  {'description': 'Multi-tenant provider tenant.',
                                                   'name': 'tenant',
                                                   'type': 'string'},
                                                  {'description': 'Multi-tenant provider site '
                                                                  'path.',
                                                   'name': 'site',
                                                   'type': 'string'},
                                                  {'description': 'Last probe or sync status.',
                                                   'name': 'last_status',
                                                   'type': 'string'},
                                                  {'description': 'Unmodified upstream route '
                                                                  'payload.',
                                                   'name': 'raw_payload',
                                                   'type': 'string'},
                                                  {'description': 'Unknown top-level record fields '
                                                                  'preserved across storage round '
                                                                  'trips.',
                                                   'name': 'extra_payload',
                                                   'type': 'string'},
                                                  {'description': 'Route discovery timestamp.',
                                                   'name': 'detected_at',
                                                   'type': 'string'}]},
                            'title': 'Board Providers'},
                           {'description': 'Stable job identities and lifecycle state.',
                            'name': 'jobs',
                            'schema': {'fields': [{'description': 'Stable normalized job identity.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Board key this job belongs to.',
                                                   'name': 'board_key',
                                                   'type': 'id'},
                                                  {'description': 'Provider adapter identifier.',
                                                   'name': 'provider_id',
                                                   'type': 'id'},
                                                  {'description': 'Provider-native job identifier.',
                                                   'name': 'remote_id',
                                                   'type': 'id'},
                                                  {'description': 'Current lifecycle status for '
                                                                  'the stable job identity.',
                                                   'name': 'status',
                                                   'type': 'string'},
                                                  {'description': 'Current normalized job version '
                                                                  'id.',
                                                   'name': 'current_version_id',
                                                   'type': 'id'},
                                                  {'description': 'Current normalized content '
                                                                  'hash.',
                                                   'name': 'current_content_hash',
                                                   'type': 'string'},
                                                  {'description': 'Current raw payload-pair hash.',
                                                   'name': 'current_payload_hash',
                                                   'type': 'string'},
                                                  {'description': 'First successful route sync '
                                                                  'that observed this job '
                                                                  'identity.',
                                                   'name': 'first_seen_at',
                                                   'type': 'datetime'},
                                                  {'description': 'Most recent successful route '
                                                                  'sync that observed this job '
                                                                  'identity.',
                                                   'name': 'last_seen_at',
                                                   'type': 'datetime'},
                                                  {'description': 'Route sync timestamp when this '
                                                                  'job disappeared while open.',
                                                   'name': 'closed_at',
                                                   'type': 'string'},
                                                  {'description': 'Last successful lifecycle '
                                                                  'update timestamp.',
                                                   'name': 'synced_at',
                                                   'type': 'datetime'},
                                                  {'description': 'Unknown top-level identity '
                                                                  'fields preserved across storage '
                                                                  'round trips.',
                                                   'name': 'extra_payload',
                                                   'type': 'string'}]},
                            'title': 'Jobs'},
                           {'description': 'Versioned normalized job content snapshots.',
                            'name': 'job_versions',
                            'schema': {'fields': [{'description': 'Job version id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Stable job id.',
                                                   'name': 'job_id',
                                                   'type': 'id'},
                                                  {'description': 'Monotonic version number.',
                                                   'name': 'version',
                                                   'type': 'integer'},
                                                  {'description': 'Normalized user-visible content '
                                                                  'hash.',
                                                   'name': 'content_hash',
                                                   'type': 'string'},
                                                  {'description': 'Raw payload-pair hash observed '
                                                                  'for version.',
                                                   'name': 'payload_hash',
                                                   'type': 'string'},
                                                  {'description': 'Public job title.',
                                                   'name': 'title',
                                                   'type': 'string'},
                                                  {'description': 'Provider-reported job '
                                                                  'locations.',
                                                   'name': 'locations',
                                                   'type': 'string'},
                                                  {'description': 'Provider-reported department.',
                                                   'name': 'department',
                                                   'type': 'string'},
                                                  {'description': 'Provider-reported team or '
                                                                  'group.',
                                                   'name': 'team',
                                                   'type': 'string'},
                                                  {'description': 'Workplace, commitment, or '
                                                                  'time-type label.',
                                                   'name': 'workplace_type',
                                                   'type': 'string'},
                                                  {'description': 'Board or company display name.',
                                                   'name': 'company',
                                                   'type': 'string'},
                                                  {'description': 'Employment or commitment type.',
                                                   'name': 'employment_type',
                                                   'type': 'string'},
                                                  {'description': 'Plain-text provider job '
                                                                  'description.',
                                                   'name': 'description',
                                                   'type': 'string'},
                                                  {'description': 'HTML provider job description.',
                                                   'name': 'description_html',
                                                   'type': 'string'},
                                                  {'description': 'JSON Resume-compatible remote '
                                                                  'work level.',
                                                   'name': 'remote',
                                                   'type': 'string'},
                                                  {'description': 'Provider compensation payload '
                                                                  'or normalized compensation '
                                                                  'details.',
                                                   'name': 'compensation',
                                                   'type': 'string'},
                                                  {'description': 'JSON Resume-compatible salary '
                                                                  'display string.',
                                                   'name': 'salary',
                                                   'type': 'string'},
                                                  {'description': 'Minimum deterministic salary or '
                                                                  'compensation value.',
                                                   'name': 'salary_min',
                                                   'type': 'numeric'},
                                                  {'description': 'Maximum deterministic salary or '
                                                                  'compensation value.',
                                                   'name': 'salary_max',
                                                   'type': 'numeric'},
                                                  {'description': 'Salary or compensation currency '
                                                                  'code.',
                                                   'name': 'salary_currency',
                                                   'type': 'string'},
                                                  {'description': 'Experience label when '
                                                                  'deterministically available.',
                                                   'name': 'experience',
                                                   'type': 'string'},
                                                  {'description': 'Deterministic responsibility '
                                                                  'bullets.',
                                                   'name': 'responsibilities',
                                                   'type': 'string'},
                                                  {'description': 'Deterministic qualification '
                                                                  'bullets.',
                                                   'name': 'qualifications',
                                                   'type': 'string'},
                                                  {'description': 'JSON Resume-compatible skill '
                                                                  'objects.',
                                                   'name': 'skills',
                                                   'type': 'string'},
                                                  {'description': 'JSON Resume-compatible '
                                                                  'job-description object.',
                                                   'name': 'job_description',
                                                   'type': 'string'},
                                                  {'description': 'Canonical public posting URL.',
                                                   'name': 'posting_url',
                                                   'type': 'url'},
                                                  {'description': 'Direct application URL.',
                                                   'name': 'apply_url',
                                                   'type': 'url'},
                                                  {'description': 'Provider-native posted '
                                                                  'timestamp.',
                                                   'name': 'posted_at',
                                                   'type': 'string'},
                                                  {'description': 'Provider-native updated '
                                                                  'timestamp.',
                                                   'name': 'updated_at',
                                                   'type': 'string'},
                                                  {'description': 'Unknown top-level version '
                                                                  'fields preserved across storage '
                                                                  'round trips.',
                                                   'name': 'extra_payload',
                                                   'type': 'string'},
                                                  {'description': 'First sync timestamp that '
                                                                  'observed this normalized '
                                                                  'content.',
                                                   'name': 'first_seen_at',
                                                   'type': 'datetime'},
                                                  {'description': 'Most recent sync timestamp that '
                                                                  'observed this normalized '
                                                                  'content.',
                                                   'name': 'last_seen_at',
                                                   'type': 'datetime'},
                                                  {'description': 'UTC timestamp when this version '
                                                                  'row was created.',
                                                   'name': 'created_at',
                                                   'type': 'datetime'}]},
                            'title': 'Job Versions'},
                           {'description': 'Indexed location labels for each normalized job '
                                           'version.',
                            'name': 'job_version_locations',
                            'schema': {'fields': [{'description': 'Stable job-version location row '
                                                                  'id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Job version this location '
                                                                  'belongs to.',
                                                   'name': 'job_version_id',
                                                   'type': 'id'},
                                                  {'description': 'Zero-based location order '
                                                                  'within the job version.',
                                                   'name': 'ordinal',
                                                   'type': 'integer'},
                                                  {'description': 'Location label text.',
                                                   'name': 'label',
                                                   'type': 'string'}]},
                            'title': 'Job Version Locations'},
                           {'description': 'Indexed skill groups for each normalized job version.',
                            'name': 'job_version_skills',
                            'schema': {'fields': [{'description': 'Stable job-version skill row '
                                                                  'id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Job version this skill group '
                                                                  'belongs to.',
                                                   'name': 'job_version_id',
                                                   'type': 'id'},
                                                  {'description': 'Zero-based skill order within '
                                                                  'the job version.',
                                                   'name': 'ordinal',
                                                   'type': 'integer'},
                                                  {'description': 'Skill group display name.',
                                                   'name': 'name',
                                                   'type': 'string'},
                                                  {'description': 'Skill group proficiency or '
                                                                  'level label.',
                                                   'name': 'level',
                                                   'type': 'string'}]},
                            'title': 'Job Version Skills'},
                           {'description': 'Indexed skill keywords for each normalized job version '
                                           'skill.',
                            'name': 'job_version_skill_keywords',
                            'schema': {'fields': [{'description': 'Stable job-version skill '
                                                                  'keyword row id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Skill group this keyword '
                                                                  'belongs to.',
                                                   'name': 'skill_id',
                                                   'type': 'id'},
                                                  {'description': 'Zero-based keyword order within '
                                                                  'the skill group.',
                                                   'name': 'ordinal',
                                                   'type': 'integer'},
                                                  {'description': 'Skill keyword text.',
                                                   'name': 'keyword',
                                                   'type': 'string'}]},
                            'title': 'Job Version Skill Keywords'},
                           {'description': 'Indexed responsibility and qualification bullets for '
                                           'each job version.',
                            'name': 'job_version_bullets',
                            'schema': {'fields': [{'description': 'Stable job-version bullet row '
                                                                  'id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Job version this bullet belongs '
                                                                  'to.',
                                                   'name': 'job_version_id',
                                                   'type': 'id'},
                                                  {'description': 'Bullet category, such as '
                                                                  'responsibility or '
                                                                  'qualification.',
                                                   'name': 'kind',
                                                   'type': 'string'},
                                                  {'description': 'Zero-based bullet order within '
                                                                  'its category.',
                                                   'name': 'ordinal',
                                                   'type': 'integer'},
                                                  {'description': 'Bullet text.',
                                                   'name': 'text',
                                                   'type': 'string'}]},
                            'title': 'Job Version Bullets'},
                           {'description': 'Raw upstream payload snapshots for audit and replay.',
                            'name': 'job_payload_snapshots',
                            'schema': {'fields': [{'description': 'Stable raw payload snapshot row '
                                                                  'id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Stable job identity this raw '
                                                                  'payload belongs to.',
                                                   'name': 'job_id',
                                                   'type': 'id'},
                                                  {'description': 'Raw payload source kind, such '
                                                                  'as listing or detail.',
                                                   'name': 'payload_kind',
                                                   'type': 'string'},
                                                  {'description': 'Canonical hash of the '
                                                                  'unmodified raw payload.',
                                                   'name': 'payload_hash',
                                                   'type': 'string'},
                                                  {'description': 'Unmodified upstream payload for '
                                                                  'audit and replay.',
                                                   'name': 'payload',
                                                   'type': 'string'},
                                                  {'description': 'UTC sync timestamp when this '
                                                                  'raw payload was observed.',
                                                   'name': 'observed_at',
                                                   'type': 'datetime'}]},
                            'title': 'Job Payload Snapshots'},
                           {'description': 'Provider route sync attempts and aggregate change '
                                           'counts.',
                            'name': 'job_sync_runs',
                            'schema': {'fields': [{'description': 'Stable provider route sync run '
                                                                  'id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Board key synced during this '
                                                                  'provider route run.',
                                                   'name': 'board_key',
                                                   'type': 'id'},
                                                  {'description': 'Provider route synced during '
                                                                  'this run.',
                                                   'name': 'provider_id',
                                                   'type': 'id'},
                                                  {'description': 'UTC timestamp for this provider '
                                                                  'route sync attempt.',
                                                   'name': 'synced_at',
                                                   'type': 'datetime'},
                                                  {'description': 'Whether the route sync '
                                                                  'completed.',
                                                   'name': 'success',
                                                   'type': 'boolean'},
                                                  {'description': 'Error message captured for '
                                                                  'failed route syncs.',
                                                   'name': 'error',
                                                   'type': 'string'},
                                                  {'description': 'Total jobs observed in the '
                                                                  'route sync.',
                                                   'name': 'job_count',
                                                   'type': 'integer'},
                                                  {'description': 'Jobs newly created by the route '
                                                                  'sync.',
                                                   'name': 'new_count',
                                                   'type': 'integer'},
                                                  {'description': 'Jobs observed without content '
                                                                  'changes.',
                                                   'name': 'unchanged_count',
                                                   'type': 'integer'},
                                                  {'description': 'Jobs with a new normalized '
                                                                  'content version.',
                                                   'name': 'changed_count',
                                                   'type': 'integer'},
                                                  {'description': 'Previously closed jobs reopened '
                                                                  'by the route sync.',
                                                   'name': 'reopened_count',
                                                   'type': 'integer'},
                                                  {'description': 'Previously open jobs closed by '
                                                                  'the route sync.',
                                                   'name': 'closed_count',
                                                   'type': 'integer'}]},
                            'title': 'Job Sync Runs'},
                           {'description': 'Per-job observations recorded during provider route '
                                           'syncs.',
                            'name': 'job_sync_observations',
                            'schema': {'fields': [{'description': 'Stable sync observation row id.',
                                                   'name': 'id',
                                                   'type': 'id'},
                                                  {'description': 'Route sync run that recorded '
                                                                  'this observation.',
                                                   'name': 'sync_run_id',
                                                   'type': 'id'},
                                                  {'description': 'Stable job identity observed '
                                                                  'during sync.',
                                                   'name': 'job_id',
                                                   'type': 'id'},
                                                  {'description': 'Normalized job version '
                                                                  'associated with this '
                                                                  'observation.',
                                                   'name': 'job_version_id',
                                                   'type': 'id'},
                                                  {'description': 'Observation category, such as '
                                                                  'new, unchanged, changed, '
                                                                  'reopened, or closed.',
                                                   'name': 'observation_kind',
                                                   'type': 'string'},
                                                  {'description': 'Normalized content hash '
                                                                  'observed during sync.',
                                                   'name': 'content_hash',
                                                   'type': 'string'},
                                                  {'description': 'Raw payload-pair hash observed '
                                                                  'during sync.',
                                                   'name': 'payload_hash',
                                                   'type': 'string'},
                                                  {'description': 'UTC timestamp when the '
                                                                  'observation was recorded.',
                                                   'name': 'observed_at',
                                                   'type': 'datetime'}]},
                            'title': 'Job Sync Observations'},
                           {'description': 'In-database table labels and descriptions for '
                                           'openoppsdb.sqlite.',
                            'name': 'openopps_tables',
                            'schema': {'fields': [{'description': 'SQLite table name.',
                                                   'name': 'table_name',
                                                   'type': 'string'},
                                                  {'description': 'Human-readable table label.',
                                                   'name': 'table_title',
                                                   'type': 'string'},
                                                  {'description': 'Plain-language table '
                                                                  'description.',
                                                   'name': 'table_description',
                                                   'type': 'string'},
                                                  {'description': 'CSV export path for this SQLite '
                                                                  'table.',
                                                   'name': 'csv_path',
                                                   'type': 'string'},
                                                  {'description': 'Parquet export path for this '
                                                                  'SQLite table.',
                                                   'name': 'parquet_path',
                                                   'type': 'string'}]},
                            'title': 'Openopps Tables'},
                           {'description': 'In-database column labels, descriptions, and schema '
                                           'hints for openoppsdb.sqlite.',
                            'name': 'openopps_columns',
                            'schema': {'fields': [{'description': 'SQLite table that owns this '
                                                                  'column.',
                                                   'name': 'table_name',
                                                   'type': 'string'},
                                                  {'description': 'SQLite column name.',
                                                   'name': 'column_name',
                                                   'type': 'string'},
                                                  {'description': 'Human-readable column label.',
                                                   'name': 'column_title',
                                                   'type': 'string'},
                                                  {'description': 'Plain-language column '
                                                                  'description.',
                                                   'name': 'column_description',
                                                   'type': 'string'},
                                                  {'description': 'Python or typing-level logical '
                                                                  'type label.',
                                                   'name': 'logical_type',
                                                   'type': 'string'},
                                                  {'description': 'JSON Schema type derived from '
                                                                  'the model field.',
                                                   'name': 'json_schema_type',
                                                   'type': 'string'},
                                                  {'description': 'Whether the source model marks '
                                                                  'the column as required.',
                                                   'name': 'required',
                                                   'type': 'boolean'},
                                                  {'description': 'Original source alias when it '
                                                                  'differs from the column name.',
                                                   'name': 'source_name',
                                                   'type': 'string'},
                                                  {'description': 'JSON Schema format hint, when '
                                                                  'available.',
                                                   'name': 'format',
                                                   'type': 'string'},
                                                  {'description': 'JSON array of allowed values, '
                                                                  'when available.',
                                                   'name': 'enum_json',
                                                   'type': 'string'},
                                                  {'description': 'JSON array of example values, '
                                                                  'when available.',
                                                   'name': 'examples_json',
                                                   'type': 'string'},
                                                  {'description': 'JSON-encoded default value, '
                                                                  'when available.',
                                                   'name': 'default_json',
                                                   'type': 'string'}]},
                            'title': 'Openopps Columns'}],
                'title': 'Openopps Database'},
               {'description': 'Full CSV table export for Durable source catalogs that discover '
                               'company boards.',
                'name': 'sources_csv',
                'path': 'exports/csv/sources.csv',
                'schema': {'fields': [{'description': 'Stable local source key.',
                                       'name': 'key',
                                       'type': 'string'},
                                      {'description': 'Canonical source URL or synthetic manual '
                                                      'source URI.',
                                       'name': 'url',
                                       'type': 'string'},
                                      {'description': 'Source adapter identifier.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'Whether unscoped syncs include this source.',
                                       'name': 'enabled',
                                       'type': 'boolean'},
                                      {'description': 'Provider version metadata.',
                                       'name': 'version',
                                       'type': 'string'},
                                      {'description': 'Source configuration and sync metadata.',
                                       'name': 'raw_metadata',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level record fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'Last successful source sync timestamp.',
                                       'name': 'synced_at',
                                       'type': 'string'}]},
                'title': 'Sources Csv'},
               {'description': 'Full CSV table export for Durable normalized company or '
                               'organization hiring boards.',
                'name': 'boards_csv',
                'path': 'exports/csv/boards.csv',
                'schema': {'fields': [{'description': 'Stable normalized board key.',
                                       'name': 'key',
                                       'type': 'string'},
                                      {'description': 'Source key that emitted this board.',
                                       'name': 'source_key',
                                       'type': 'id'},
                                      {'description': 'All sources that currently contain this '
                                                      'board domain.',
                                       'name': 'source_keys',
                                       'type': 'string'},
                                      {'description': 'Source-specific emitted board keys merged '
                                                      'into this board.',
                                       'name': 'source_board_keys',
                                       'type': 'string'},
                                      {'description': 'Provider-native board identifier.',
                                       'name': 'remote_id',
                                       'type': 'id'},
                                      {'description': 'Provider-native slug.',
                                       'name': 'remote_slug',
                                       'type': 'string'},
                                      {'description': 'Company or board display name.',
                                       'name': 'name',
                                       'type': 'string'},
                                      {'description': 'Normalized website domain.',
                                       'name': 'domain',
                                       'type': 'string'},
                                      {'description': 'Canonical company website URL.',
                                       'name': 'website_url',
                                       'type': 'url'},
                                      {'description': 'Provider-supplied board description.',
                                       'name': 'description',
                                       'type': 'string'},
                                      {'description': 'Industry or market tags.',
                                       'name': 'markets',
                                       'type': 'string'},
                                      {'description': 'Office or hiring locations.',
                                       'name': 'locations',
                                       'type': 'string'},
                                      {'description': 'Employee or team-size estimate.',
                                       'name': 'staff_count',
                                       'type': 'integer'},
                                      {'description': 'Approximate number of open jobs.',
                                       'name': 'num_jobs_hint',
                                       'type': 'integer'},
                                      {'description': 'Unmodified upstream board payload.',
                                       'name': 'raw_payload',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level record fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'Last successful board sync timestamp.',
                                       'name': 'synced_at',
                                       'type': 'string'}]},
                'title': 'Boards Csv'},
               {'description': 'Full CSV table export for Durable provider routes that connect '
                               'boards to upstream systems.',
                'name': 'board_providers_csv',
                'path': 'exports/csv/board_providers.csv',
                'schema': {'fields': [{'description': 'Stable route primary key.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Source key that reported this route.',
                                       'name': 'source_key',
                                       'type': 'id'},
                                      {'description': 'Board key this route belongs to.',
                                       'name': 'board_key',
                                       'type': 'id'},
                                      {'description': 'Provider adapter identifier.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'Human-readable upstream route label.',
                                       'name': 'label',
                                       'type': 'string'},
                                      {'description': 'Normalized provider support level.',
                                       'name': 'support_level',
                                       'type': 'string'},
                                      {'description': 'Approximate provider-reported job count.',
                                       'name': 'count_hint',
                                       'type': 'integer'},
                                      {'description': 'Hosted job board URL.',
                                       'name': 'board_url',
                                       'type': 'url'},
                                      {'description': 'Provider-specific board token or slug.',
                                       'name': 'token',
                                       'type': 'string'},
                                      {'description': 'Multi-tenant provider host.',
                                       'name': 'host',
                                       'type': 'string'},
                                      {'description': 'Multi-tenant provider tenant.',
                                       'name': 'tenant',
                                       'type': 'string'},
                                      {'description': 'Multi-tenant provider site path.',
                                       'name': 'site',
                                       'type': 'string'},
                                      {'description': 'Last probe or sync status.',
                                       'name': 'last_status',
                                       'type': 'string'},
                                      {'description': 'Unmodified upstream route payload.',
                                       'name': 'raw_payload',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level record fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'Route discovery timestamp.',
                                       'name': 'detected_at',
                                       'type': 'string'}]},
                'title': 'Board Providers Csv'},
               {'description': 'Full CSV table export for Stable job identities and lifecycle '
                               'state.',
                'name': 'jobs_csv',
                'path': 'exports/csv/jobs.csv',
                'schema': {'fields': [{'description': 'Stable normalized job identity.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Board key this job belongs to.',
                                       'name': 'board_key',
                                       'type': 'id'},
                                      {'description': 'Provider adapter identifier.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'Provider-native job identifier.',
                                       'name': 'remote_id',
                                       'type': 'id'},
                                      {'description': 'Current lifecycle status for the stable job '
                                                      'identity.',
                                       'name': 'status',
                                       'type': 'string'},
                                      {'description': 'Current normalized job version id.',
                                       'name': 'current_version_id',
                                       'type': 'id'},
                                      {'description': 'Current normalized content hash.',
                                       'name': 'current_content_hash',
                                       'type': 'string'},
                                      {'description': 'Current raw payload-pair hash.',
                                       'name': 'current_payload_hash',
                                       'type': 'string'},
                                      {'description': 'First successful route sync that observed '
                                                      'this job identity.',
                                       'name': 'first_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'Most recent successful route sync that '
                                                      'observed this job identity.',
                                       'name': 'last_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'Route sync timestamp when this job '
                                                      'disappeared while open.',
                                       'name': 'closed_at',
                                       'type': 'string'},
                                      {'description': 'Last successful lifecycle update timestamp.',
                                       'name': 'synced_at',
                                       'type': 'datetime'},
                                      {'description': 'Unknown top-level identity fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'}]},
                'title': 'Jobs Csv'},
               {'description': 'Full CSV table export for Versioned normalized job content '
                               'snapshots.',
                'name': 'job_versions_csv',
                'path': 'exports/csv/job_versions.csv',
                'schema': {'fields': [{'description': 'Job version id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Stable job id.',
                                       'name': 'job_id',
                                       'type': 'id'},
                                      {'description': 'Monotonic version number.',
                                       'name': 'version',
                                       'type': 'integer'},
                                      {'description': 'Normalized user-visible content hash.',
                                       'name': 'content_hash',
                                       'type': 'string'},
                                      {'description': 'Raw payload-pair hash observed for version.',
                                       'name': 'payload_hash',
                                       'type': 'string'},
                                      {'description': 'Public job title.',
                                       'name': 'title',
                                       'type': 'string'},
                                      {'description': 'Provider-reported job locations.',
                                       'name': 'locations',
                                       'type': 'string'},
                                      {'description': 'Provider-reported department.',
                                       'name': 'department',
                                       'type': 'string'},
                                      {'description': 'Provider-reported team or group.',
                                       'name': 'team',
                                       'type': 'string'},
                                      {'description': 'Workplace, commitment, or time-type label.',
                                       'name': 'workplace_type',
                                       'type': 'string'},
                                      {'description': 'Board or company display name.',
                                       'name': 'company',
                                       'type': 'string'},
                                      {'description': 'Employment or commitment type.',
                                       'name': 'employment_type',
                                       'type': 'string'},
                                      {'description': 'Plain-text provider job description.',
                                       'name': 'description',
                                       'type': 'string'},
                                      {'description': 'HTML provider job description.',
                                       'name': 'description_html',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible remote work level.',
                                       'name': 'remote',
                                       'type': 'string'},
                                      {'description': 'Provider compensation payload or normalized '
                                                      'compensation details.',
                                       'name': 'compensation',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible salary display '
                                                      'string.',
                                       'name': 'salary',
                                       'type': 'string'},
                                      {'description': 'Minimum deterministic salary or '
                                                      'compensation value.',
                                       'name': 'salary_min',
                                       'type': 'numeric'},
                                      {'description': 'Maximum deterministic salary or '
                                                      'compensation value.',
                                       'name': 'salary_max',
                                       'type': 'numeric'},
                                      {'description': 'Salary or compensation currency code.',
                                       'name': 'salary_currency',
                                       'type': 'string'},
                                      {'description': 'Experience label when deterministically '
                                                      'available.',
                                       'name': 'experience',
                                       'type': 'string'},
                                      {'description': 'Deterministic responsibility bullets.',
                                       'name': 'responsibilities',
                                       'type': 'string'},
                                      {'description': 'Deterministic qualification bullets.',
                                       'name': 'qualifications',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible skill objects.',
                                       'name': 'skills',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible job-description '
                                                      'object.',
                                       'name': 'job_description',
                                       'type': 'string'},
                                      {'description': 'Canonical public posting URL.',
                                       'name': 'posting_url',
                                       'type': 'url'},
                                      {'description': 'Direct application URL.',
                                       'name': 'apply_url',
                                       'type': 'url'},
                                      {'description': 'Provider-native posted timestamp.',
                                       'name': 'posted_at',
                                       'type': 'string'},
                                      {'description': 'Provider-native updated timestamp.',
                                       'name': 'updated_at',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level version fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'First sync timestamp that observed this '
                                                      'normalized content.',
                                       'name': 'first_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'Most recent sync timestamp that observed '
                                                      'this normalized content.',
                                       'name': 'last_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'UTC timestamp when this version row was '
                                                      'created.',
                                       'name': 'created_at',
                                       'type': 'datetime'}]},
                'title': 'Job Versions Csv'},
               {'description': 'Full CSV table export for Indexed location labels for each '
                               'normalized job version.',
                'name': 'job_version_locations_csv',
                'path': 'exports/csv/job_version_locations.csv',
                'schema': {'fields': [{'description': 'Stable job-version location row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Job version this location belongs to.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Zero-based location order within the job '
                                                      'version.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Location label text.',
                                       'name': 'label',
                                       'type': 'string'}]},
                'title': 'Job Version Locations Csv'},
               {'description': 'Full CSV table export for Indexed skill groups for each normalized '
                               'job version.',
                'name': 'job_version_skills_csv',
                'path': 'exports/csv/job_version_skills.csv',
                'schema': {'fields': [{'description': 'Stable job-version skill row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Job version this skill group belongs to.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Zero-based skill order within the job '
                                                      'version.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Skill group display name.',
                                       'name': 'name',
                                       'type': 'string'},
                                      {'description': 'Skill group proficiency or level label.',
                                       'name': 'level',
                                       'type': 'string'}]},
                'title': 'Job Version Skills Csv'},
               {'description': 'Full CSV table export for Indexed skill keywords for each '
                               'normalized job version skill.',
                'name': 'job_version_skill_keywords_csv',
                'path': 'exports/csv/job_version_skill_keywords.csv',
                'schema': {'fields': [{'description': 'Stable job-version skill keyword row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Skill group this keyword belongs to.',
                                       'name': 'skill_id',
                                       'type': 'id'},
                                      {'description': 'Zero-based keyword order within the skill '
                                                      'group.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Skill keyword text.',
                                       'name': 'keyword',
                                       'type': 'string'}]},
                'title': 'Job Version Skill Keywords Csv'},
               {'description': 'Full CSV table export for Indexed responsibility and qualification '
                               'bullets for each job version.',
                'name': 'job_version_bullets_csv',
                'path': 'exports/csv/job_version_bullets.csv',
                'schema': {'fields': [{'description': 'Stable job-version bullet row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Job version this bullet belongs to.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Bullet category, such as responsibility or '
                                                      'qualification.',
                                       'name': 'kind',
                                       'type': 'string'},
                                      {'description': 'Zero-based bullet order within its '
                                                      'category.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Bullet text.',
                                       'name': 'text',
                                       'type': 'string'}]},
                'title': 'Job Version Bullets Csv'},
               {'description': 'Full CSV table export for Raw upstream payload snapshots for audit '
                               'and replay.',
                'name': 'job_payload_snapshots_csv',
                'path': 'exports/csv/job_payload_snapshots.csv',
                'schema': {'fields': [{'description': 'Stable raw payload snapshot row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Stable job identity this raw payload '
                                                      'belongs to.',
                                       'name': 'job_id',
                                       'type': 'id'},
                                      {'description': 'Raw payload source kind, such as listing or '
                                                      'detail.',
                                       'name': 'payload_kind',
                                       'type': 'string'},
                                      {'description': 'Canonical hash of the unmodified raw '
                                                      'payload.',
                                       'name': 'payload_hash',
                                       'type': 'string'},
                                      {'description': 'Unmodified upstream payload for audit and '
                                                      'replay.',
                                       'name': 'payload',
                                       'type': 'string'},
                                      {'description': 'UTC sync timestamp when this raw payload '
                                                      'was observed.',
                                       'name': 'observed_at',
                                       'type': 'datetime'}]},
                'title': 'Job Payload Snapshots Csv'},
               {'description': 'Full CSV table export for Provider route sync attempts and '
                               'aggregate change counts.',
                'name': 'job_sync_runs_csv',
                'path': 'exports/csv/job_sync_runs.csv',
                'schema': {'fields': [{'description': 'Stable provider route sync run id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Board key synced during this provider route '
                                                      'run.',
                                       'name': 'board_key',
                                       'type': 'id'},
                                      {'description': 'Provider route synced during this run.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'UTC timestamp for this provider route sync '
                                                      'attempt.',
                                       'name': 'synced_at',
                                       'type': 'datetime'},
                                      {'description': 'Whether the route sync completed.',
                                       'name': 'success',
                                       'type': 'boolean'},
                                      {'description': 'Error message captured for failed route '
                                                      'syncs.',
                                       'name': 'error',
                                       'type': 'string'},
                                      {'description': 'Total jobs observed in the route sync.',
                                       'name': 'job_count',
                                       'type': 'integer'},
                                      {'description': 'Jobs newly created by the route sync.',
                                       'name': 'new_count',
                                       'type': 'integer'},
                                      {'description': 'Jobs observed without content changes.',
                                       'name': 'unchanged_count',
                                       'type': 'integer'},
                                      {'description': 'Jobs with a new normalized content version.',
                                       'name': 'changed_count',
                                       'type': 'integer'},
                                      {'description': 'Previously closed jobs reopened by the '
                                                      'route sync.',
                                       'name': 'reopened_count',
                                       'type': 'integer'},
                                      {'description': 'Previously open jobs closed by the route '
                                                      'sync.',
                                       'name': 'closed_count',
                                       'type': 'integer'}]},
                'title': 'Job Sync Runs Csv'},
               {'description': 'Full CSV table export for Per-job observations recorded during '
                               'provider route syncs.',
                'name': 'job_sync_observations_csv',
                'path': 'exports/csv/job_sync_observations.csv',
                'schema': {'fields': [{'description': 'Stable sync observation row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Route sync run that recorded this '
                                                      'observation.',
                                       'name': 'sync_run_id',
                                       'type': 'id'},
                                      {'description': 'Stable job identity observed during sync.',
                                       'name': 'job_id',
                                       'type': 'id'},
                                      {'description': 'Normalized job version associated with this '
                                                      'observation.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Observation category, such as new, '
                                                      'unchanged, changed, reopened, or closed.',
                                       'name': 'observation_kind',
                                       'type': 'string'},
                                      {'description': 'Normalized content hash observed during '
                                                      'sync.',
                                       'name': 'content_hash',
                                       'type': 'string'},
                                      {'description': 'Raw payload-pair hash observed during sync.',
                                       'name': 'payload_hash',
                                       'type': 'string'},
                                      {'description': 'UTC timestamp when the observation was '
                                                      'recorded.',
                                       'name': 'observed_at',
                                       'type': 'datetime'}]},
                'title': 'Job Sync Observations Csv'},
               {'description': 'Full CSV table export for In-database table labels and '
                               'descriptions for openoppsdb.sqlite.',
                'name': 'openopps_tables_csv',
                'path': 'exports/csv/openopps_tables.csv',
                'schema': {'fields': [{'description': 'SQLite table name.',
                                       'name': 'table_name',
                                       'type': 'string'},
                                      {'description': 'Human-readable table label.',
                                       'name': 'table_title',
                                       'type': 'string'},
                                      {'description': 'Plain-language table description.',
                                       'name': 'table_description',
                                       'type': 'string'},
                                      {'description': 'CSV export path for this SQLite table.',
                                       'name': 'csv_path',
                                       'type': 'string'},
                                      {'description': 'Parquet export path for this SQLite table.',
                                       'name': 'parquet_path',
                                       'type': 'string'}]},
                'title': 'Openopps Tables Csv'},
               {'description': 'Full CSV table export for In-database column labels, descriptions, '
                               'and schema hints for openoppsdb.sqlite.',
                'name': 'openopps_columns_csv',
                'path': 'exports/csv/openopps_columns.csv',
                'schema': {'fields': [{'description': 'SQLite table that owns this column.',
                                       'name': 'table_name',
                                       'type': 'string'},
                                      {'description': 'SQLite column name.',
                                       'name': 'column_name',
                                       'type': 'string'},
                                      {'description': 'Human-readable column label.',
                                       'name': 'column_title',
                                       'type': 'string'},
                                      {'description': 'Plain-language column description.',
                                       'name': 'column_description',
                                       'type': 'string'},
                                      {'description': 'Python or typing-level logical type label.',
                                       'name': 'logical_type',
                                       'type': 'string'},
                                      {'description': 'JSON Schema type derived from the model '
                                                      'field.',
                                       'name': 'json_schema_type',
                                       'type': 'string'},
                                      {'description': 'Whether the source model marks the column '
                                                      'as required.',
                                       'name': 'required',
                                       'type': 'boolean'},
                                      {'description': 'Original source alias when it differs from '
                                                      'the column name.',
                                       'name': 'source_name',
                                       'type': 'string'},
                                      {'description': 'JSON Schema format hint, when available.',
                                       'name': 'format',
                                       'type': 'string'},
                                      {'description': 'JSON array of allowed values, when '
                                                      'available.',
                                       'name': 'enum_json',
                                       'type': 'string'},
                                      {'description': 'JSON array of example values, when '
                                                      'available.',
                                       'name': 'examples_json',
                                       'type': 'string'},
                                      {'description': 'JSON-encoded default value, when available.',
                                       'name': 'default_json',
                                       'type': 'string'}]},
                'title': 'Openopps Columns Csv'},
               {'description': 'Full Parquet table export for Durable source catalogs that '
                               'discover company boards.',
                'name': 'sources_parquet',
                'path': 'exports/parquet/sources.parquet',
                'schema': {'fields': [{'description': 'Stable local source key.',
                                       'name': 'key',
                                       'type': 'string'},
                                      {'description': 'Canonical source URL or synthetic manual '
                                                      'source URI.',
                                       'name': 'url',
                                       'type': 'string'},
                                      {'description': 'Source adapter identifier.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'Whether unscoped syncs include this source.',
                                       'name': 'enabled',
                                       'type': 'boolean'},
                                      {'description': 'Provider version metadata.',
                                       'name': 'version',
                                       'type': 'string'},
                                      {'description': 'Source configuration and sync metadata.',
                                       'name': 'raw_metadata',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level record fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'Last successful source sync timestamp.',
                                       'name': 'synced_at',
                                       'type': 'string'}]},
                'title': 'Sources Parquet'},
               {'description': 'Full Parquet table export for Durable normalized company or '
                               'organization hiring boards.',
                'name': 'boards_parquet',
                'path': 'exports/parquet/boards.parquet',
                'schema': {'fields': [{'description': 'Stable normalized board key.',
                                       'name': 'key',
                                       'type': 'string'},
                                      {'description': 'Source key that emitted this board.',
                                       'name': 'source_key',
                                       'type': 'id'},
                                      {'description': 'All sources that currently contain this '
                                                      'board domain.',
                                       'name': 'source_keys',
                                       'type': 'string'},
                                      {'description': 'Source-specific emitted board keys merged '
                                                      'into this board.',
                                       'name': 'source_board_keys',
                                       'type': 'string'},
                                      {'description': 'Provider-native board identifier.',
                                       'name': 'remote_id',
                                       'type': 'id'},
                                      {'description': 'Provider-native slug.',
                                       'name': 'remote_slug',
                                       'type': 'string'},
                                      {'description': 'Company or board display name.',
                                       'name': 'name',
                                       'type': 'string'},
                                      {'description': 'Normalized website domain.',
                                       'name': 'domain',
                                       'type': 'string'},
                                      {'description': 'Canonical company website URL.',
                                       'name': 'website_url',
                                       'type': 'url'},
                                      {'description': 'Provider-supplied board description.',
                                       'name': 'description',
                                       'type': 'string'},
                                      {'description': 'Industry or market tags.',
                                       'name': 'markets',
                                       'type': 'string'},
                                      {'description': 'Office or hiring locations.',
                                       'name': 'locations',
                                       'type': 'string'},
                                      {'description': 'Employee or team-size estimate.',
                                       'name': 'staff_count',
                                       'type': 'integer'},
                                      {'description': 'Approximate number of open jobs.',
                                       'name': 'num_jobs_hint',
                                       'type': 'integer'},
                                      {'description': 'Unmodified upstream board payload.',
                                       'name': 'raw_payload',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level record fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'Last successful board sync timestamp.',
                                       'name': 'synced_at',
                                       'type': 'string'}]},
                'title': 'Boards Parquet'},
               {'description': 'Full Parquet table export for Durable provider routes that connect '
                               'boards to upstream systems.',
                'name': 'board_providers_parquet',
                'path': 'exports/parquet/board_providers.parquet',
                'schema': {'fields': [{'description': 'Stable route primary key.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Source key that reported this route.',
                                       'name': 'source_key',
                                       'type': 'id'},
                                      {'description': 'Board key this route belongs to.',
                                       'name': 'board_key',
                                       'type': 'id'},
                                      {'description': 'Provider adapter identifier.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'Human-readable upstream route label.',
                                       'name': 'label',
                                       'type': 'string'},
                                      {'description': 'Normalized provider support level.',
                                       'name': 'support_level',
                                       'type': 'string'},
                                      {'description': 'Approximate provider-reported job count.',
                                       'name': 'count_hint',
                                       'type': 'integer'},
                                      {'description': 'Hosted job board URL.',
                                       'name': 'board_url',
                                       'type': 'url'},
                                      {'description': 'Provider-specific board token or slug.',
                                       'name': 'token',
                                       'type': 'string'},
                                      {'description': 'Multi-tenant provider host.',
                                       'name': 'host',
                                       'type': 'string'},
                                      {'description': 'Multi-tenant provider tenant.',
                                       'name': 'tenant',
                                       'type': 'string'},
                                      {'description': 'Multi-tenant provider site path.',
                                       'name': 'site',
                                       'type': 'string'},
                                      {'description': 'Last probe or sync status.',
                                       'name': 'last_status',
                                       'type': 'string'},
                                      {'description': 'Unmodified upstream route payload.',
                                       'name': 'raw_payload',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level record fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'Route discovery timestamp.',
                                       'name': 'detected_at',
                                       'type': 'string'}]},
                'title': 'Board Providers Parquet'},
               {'description': 'Full Parquet table export for Stable job identities and lifecycle '
                               'state.',
                'name': 'jobs_parquet',
                'path': 'exports/parquet/jobs.parquet',
                'schema': {'fields': [{'description': 'Stable normalized job identity.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Board key this job belongs to.',
                                       'name': 'board_key',
                                       'type': 'id'},
                                      {'description': 'Provider adapter identifier.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'Provider-native job identifier.',
                                       'name': 'remote_id',
                                       'type': 'id'},
                                      {'description': 'Current lifecycle status for the stable job '
                                                      'identity.',
                                       'name': 'status',
                                       'type': 'string'},
                                      {'description': 'Current normalized job version id.',
                                       'name': 'current_version_id',
                                       'type': 'id'},
                                      {'description': 'Current normalized content hash.',
                                       'name': 'current_content_hash',
                                       'type': 'string'},
                                      {'description': 'Current raw payload-pair hash.',
                                       'name': 'current_payload_hash',
                                       'type': 'string'},
                                      {'description': 'First successful route sync that observed '
                                                      'this job identity.',
                                       'name': 'first_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'Most recent successful route sync that '
                                                      'observed this job identity.',
                                       'name': 'last_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'Route sync timestamp when this job '
                                                      'disappeared while open.',
                                       'name': 'closed_at',
                                       'type': 'string'},
                                      {'description': 'Last successful lifecycle update timestamp.',
                                       'name': 'synced_at',
                                       'type': 'datetime'},
                                      {'description': 'Unknown top-level identity fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'}]},
                'title': 'Jobs Parquet'},
               {'description': 'Full Parquet table export for Versioned normalized job content '
                               'snapshots.',
                'name': 'job_versions_parquet',
                'path': 'exports/parquet/job_versions.parquet',
                'schema': {'fields': [{'description': 'Job version id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Stable job id.',
                                       'name': 'job_id',
                                       'type': 'id'},
                                      {'description': 'Monotonic version number.',
                                       'name': 'version',
                                       'type': 'integer'},
                                      {'description': 'Normalized user-visible content hash.',
                                       'name': 'content_hash',
                                       'type': 'string'},
                                      {'description': 'Raw payload-pair hash observed for version.',
                                       'name': 'payload_hash',
                                       'type': 'string'},
                                      {'description': 'Public job title.',
                                       'name': 'title',
                                       'type': 'string'},
                                      {'description': 'Provider-reported job locations.',
                                       'name': 'locations',
                                       'type': 'string'},
                                      {'description': 'Provider-reported department.',
                                       'name': 'department',
                                       'type': 'string'},
                                      {'description': 'Provider-reported team or group.',
                                       'name': 'team',
                                       'type': 'string'},
                                      {'description': 'Workplace, commitment, or time-type label.',
                                       'name': 'workplace_type',
                                       'type': 'string'},
                                      {'description': 'Board or company display name.',
                                       'name': 'company',
                                       'type': 'string'},
                                      {'description': 'Employment or commitment type.',
                                       'name': 'employment_type',
                                       'type': 'string'},
                                      {'description': 'Plain-text provider job description.',
                                       'name': 'description',
                                       'type': 'string'},
                                      {'description': 'HTML provider job description.',
                                       'name': 'description_html',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible remote work level.',
                                       'name': 'remote',
                                       'type': 'string'},
                                      {'description': 'Provider compensation payload or normalized '
                                                      'compensation details.',
                                       'name': 'compensation',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible salary display '
                                                      'string.',
                                       'name': 'salary',
                                       'type': 'string'},
                                      {'description': 'Minimum deterministic salary or '
                                                      'compensation value.',
                                       'name': 'salary_min',
                                       'type': 'numeric'},
                                      {'description': 'Maximum deterministic salary or '
                                                      'compensation value.',
                                       'name': 'salary_max',
                                       'type': 'numeric'},
                                      {'description': 'Salary or compensation currency code.',
                                       'name': 'salary_currency',
                                       'type': 'string'},
                                      {'description': 'Experience label when deterministically '
                                                      'available.',
                                       'name': 'experience',
                                       'type': 'string'},
                                      {'description': 'Deterministic responsibility bullets.',
                                       'name': 'responsibilities',
                                       'type': 'string'},
                                      {'description': 'Deterministic qualification bullets.',
                                       'name': 'qualifications',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible skill objects.',
                                       'name': 'skills',
                                       'type': 'string'},
                                      {'description': 'JSON Resume-compatible job-description '
                                                      'object.',
                                       'name': 'job_description',
                                       'type': 'string'},
                                      {'description': 'Canonical public posting URL.',
                                       'name': 'posting_url',
                                       'type': 'url'},
                                      {'description': 'Direct application URL.',
                                       'name': 'apply_url',
                                       'type': 'url'},
                                      {'description': 'Provider-native posted timestamp.',
                                       'name': 'posted_at',
                                       'type': 'string'},
                                      {'description': 'Provider-native updated timestamp.',
                                       'name': 'updated_at',
                                       'type': 'string'},
                                      {'description': 'Unknown top-level version fields preserved '
                                                      'across storage round trips.',
                                       'name': 'extra_payload',
                                       'type': 'string'},
                                      {'description': 'First sync timestamp that observed this '
                                                      'normalized content.',
                                       'name': 'first_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'Most recent sync timestamp that observed '
                                                      'this normalized content.',
                                       'name': 'last_seen_at',
                                       'type': 'datetime'},
                                      {'description': 'UTC timestamp when this version row was '
                                                      'created.',
                                       'name': 'created_at',
                                       'type': 'datetime'}]},
                'title': 'Job Versions Parquet'},
               {'description': 'Full Parquet table export for Indexed location labels for each '
                               'normalized job version.',
                'name': 'job_version_locations_parquet',
                'path': 'exports/parquet/job_version_locations.parquet',
                'schema': {'fields': [{'description': 'Stable job-version location row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Job version this location belongs to.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Zero-based location order within the job '
                                                      'version.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Location label text.',
                                       'name': 'label',
                                       'type': 'string'}]},
                'title': 'Job Version Locations Parquet'},
               {'description': 'Full Parquet table export for Indexed skill groups for each '
                               'normalized job version.',
                'name': 'job_version_skills_parquet',
                'path': 'exports/parquet/job_version_skills.parquet',
                'schema': {'fields': [{'description': 'Stable job-version skill row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Job version this skill group belongs to.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Zero-based skill order within the job '
                                                      'version.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Skill group display name.',
                                       'name': 'name',
                                       'type': 'string'},
                                      {'description': 'Skill group proficiency or level label.',
                                       'name': 'level',
                                       'type': 'string'}]},
                'title': 'Job Version Skills Parquet'},
               {'description': 'Full Parquet table export for Indexed skill keywords for each '
                               'normalized job version skill.',
                'name': 'job_version_skill_keywords_parquet',
                'path': 'exports/parquet/job_version_skill_keywords.parquet',
                'schema': {'fields': [{'description': 'Stable job-version skill keyword row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Skill group this keyword belongs to.',
                                       'name': 'skill_id',
                                       'type': 'id'},
                                      {'description': 'Zero-based keyword order within the skill '
                                                      'group.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Skill keyword text.',
                                       'name': 'keyword',
                                       'type': 'string'}]},
                'title': 'Job Version Skill Keywords Parquet'},
               {'description': 'Full Parquet table export for Indexed responsibility and '
                               'qualification bullets for each job version.',
                'name': 'job_version_bullets_parquet',
                'path': 'exports/parquet/job_version_bullets.parquet',
                'schema': {'fields': [{'description': 'Stable job-version bullet row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Job version this bullet belongs to.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Bullet category, such as responsibility or '
                                                      'qualification.',
                                       'name': 'kind',
                                       'type': 'string'},
                                      {'description': 'Zero-based bullet order within its '
                                                      'category.',
                                       'name': 'ordinal',
                                       'type': 'integer'},
                                      {'description': 'Bullet text.',
                                       'name': 'text',
                                       'type': 'string'}]},
                'title': 'Job Version Bullets Parquet'},
               {'description': 'Full Parquet table export for Raw upstream payload snapshots for '
                               'audit and replay.',
                'name': 'job_payload_snapshots_parquet',
                'path': 'exports/parquet/job_payload_snapshots.parquet',
                'schema': {'fields': [{'description': 'Stable raw payload snapshot row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Stable job identity this raw payload '
                                                      'belongs to.',
                                       'name': 'job_id',
                                       'type': 'id'},
                                      {'description': 'Raw payload source kind, such as listing or '
                                                      'detail.',
                                       'name': 'payload_kind',
                                       'type': 'string'},
                                      {'description': 'Canonical hash of the unmodified raw '
                                                      'payload.',
                                       'name': 'payload_hash',
                                       'type': 'string'},
                                      {'description': 'Unmodified upstream payload for audit and '
                                                      'replay.',
                                       'name': 'payload',
                                       'type': 'string'},
                                      {'description': 'UTC sync timestamp when this raw payload '
                                                      'was observed.',
                                       'name': 'observed_at',
                                       'type': 'datetime'}]},
                'title': 'Job Payload Snapshots Parquet'},
               {'description': 'Full Parquet table export for Provider route sync attempts and '
                               'aggregate change counts.',
                'name': 'job_sync_runs_parquet',
                'path': 'exports/parquet/job_sync_runs.parquet',
                'schema': {'fields': [{'description': 'Stable provider route sync run id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Board key synced during this provider route '
                                                      'run.',
                                       'name': 'board_key',
                                       'type': 'id'},
                                      {'description': 'Provider route synced during this run.',
                                       'name': 'provider_id',
                                       'type': 'id'},
                                      {'description': 'UTC timestamp for this provider route sync '
                                                      'attempt.',
                                       'name': 'synced_at',
                                       'type': 'datetime'},
                                      {'description': 'Whether the route sync completed.',
                                       'name': 'success',
                                       'type': 'boolean'},
                                      {'description': 'Error message captured for failed route '
                                                      'syncs.',
                                       'name': 'error',
                                       'type': 'string'},
                                      {'description': 'Total jobs observed in the route sync.',
                                       'name': 'job_count',
                                       'type': 'integer'},
                                      {'description': 'Jobs newly created by the route sync.',
                                       'name': 'new_count',
                                       'type': 'integer'},
                                      {'description': 'Jobs observed without content changes.',
                                       'name': 'unchanged_count',
                                       'type': 'integer'},
                                      {'description': 'Jobs with a new normalized content version.',
                                       'name': 'changed_count',
                                       'type': 'integer'},
                                      {'description': 'Previously closed jobs reopened by the '
                                                      'route sync.',
                                       'name': 'reopened_count',
                                       'type': 'integer'},
                                      {'description': 'Previously open jobs closed by the route '
                                                      'sync.',
                                       'name': 'closed_count',
                                       'type': 'integer'}]},
                'title': 'Job Sync Runs Parquet'},
               {'description': 'Full Parquet table export for Per-job observations recorded during '
                               'provider route syncs.',
                'name': 'job_sync_observations_parquet',
                'path': 'exports/parquet/job_sync_observations.parquet',
                'schema': {'fields': [{'description': 'Stable sync observation row id.',
                                       'name': 'id',
                                       'type': 'id'},
                                      {'description': 'Route sync run that recorded this '
                                                      'observation.',
                                       'name': 'sync_run_id',
                                       'type': 'id'},
                                      {'description': 'Stable job identity observed during sync.',
                                       'name': 'job_id',
                                       'type': 'id'},
                                      {'description': 'Normalized job version associated with this '
                                                      'observation.',
                                       'name': 'job_version_id',
                                       'type': 'id'},
                                      {'description': 'Observation category, such as new, '
                                                      'unchanged, changed, reopened, or closed.',
                                       'name': 'observation_kind',
                                       'type': 'string'},
                                      {'description': 'Normalized content hash observed during '
                                                      'sync.',
                                       'name': 'content_hash',
                                       'type': 'string'},
                                      {'description': 'Raw payload-pair hash observed during sync.',
                                       'name': 'payload_hash',
                                       'type': 'string'},
                                      {'description': 'UTC timestamp when the observation was '
                                                      'recorded.',
                                       'name': 'observed_at',
                                       'type': 'datetime'}]},
                'title': 'Job Sync Observations Parquet'},
               {'description': 'Full Parquet table export for In-database table labels and '
                               'descriptions for openoppsdb.sqlite.',
                'name': 'openopps_tables_parquet',
                'path': 'exports/parquet/openopps_tables.parquet',
                'schema': {'fields': [{'description': 'SQLite table name.',
                                       'name': 'table_name',
                                       'type': 'string'},
                                      {'description': 'Human-readable table label.',
                                       'name': 'table_title',
                                       'type': 'string'},
                                      {'description': 'Plain-language table description.',
                                       'name': 'table_description',
                                       'type': 'string'},
                                      {'description': 'CSV export path for this SQLite table.',
                                       'name': 'csv_path',
                                       'type': 'string'},
                                      {'description': 'Parquet export path for this SQLite table.',
                                       'name': 'parquet_path',
                                       'type': 'string'}]},
                'title': 'Openopps Tables Parquet'},
               {'description': 'Full Parquet table export for In-database column labels, '
                               'descriptions, and schema hints for openoppsdb.sqlite.',
                'name': 'openopps_columns_parquet',
                'path': 'exports/parquet/openopps_columns.parquet',
                'schema': {'fields': [{'description': 'SQLite table that owns this column.',
                                       'name': 'table_name',
                                       'type': 'string'},
                                      {'description': 'SQLite column name.',
                                       'name': 'column_name',
                                       'type': 'string'},
                                      {'description': 'Human-readable column label.',
                                       'name': 'column_title',
                                       'type': 'string'},
                                      {'description': 'Plain-language column description.',
                                       'name': 'column_description',
                                       'type': 'string'},
                                      {'description': 'Python or typing-level logical type label.',
                                       'name': 'logical_type',
                                       'type': 'string'},
                                      {'description': 'JSON Schema type derived from the model '
                                                      'field.',
                                       'name': 'json_schema_type',
                                       'type': 'string'},
                                      {'description': 'Whether the source model marks the column '
                                                      'as required.',
                                       'name': 'required',
                                       'type': 'boolean'},
                                      {'description': 'Original source alias when it differs from '
                                                      'the column name.',
                                       'name': 'source_name',
                                       'type': 'string'},
                                      {'description': 'JSON Schema format hint, when available.',
                                       'name': 'format',
                                       'type': 'string'},
                                      {'description': 'JSON array of allowed values, when '
                                                      'available.',
                                       'name': 'enum_json',
                                       'type': 'string'},
                                      {'description': 'JSON array of example values, when '
                                                      'available.',
                                       'name': 'examples_json',
                                       'type': 'string'},
                                      {'description': 'JSON-encoded default value, when available.',
                                       'name': 'default_json',
                                       'type': 'string'}]},
                'title': 'Openopps Columns Parquet'}],
 'subtitle': 'Daily SQLite, CSV, and Parquet public startup hiring-board ledger.',
 'title': 'openoppsdb',
 'userSpecifiedSources': 'Public company and startup hiring boards, public portfolio-company '
                         'directories, and provider-hosted public job posting endpoints discovered '
                         'by the OpenOpps CLI.'}
SQLITE_TABLE_METADATA = [{'description': 'Durable source catalogs that discover company boards.',
  'name': 'sources',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable local source key.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'key',
                         'required': True,
                         'title': 'Key',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Canonical source URL or synthetic manual source URI.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'url',
                         'required': True,
                         'title': 'Url',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Source adapter identifier.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'provider_id',
                         'required': True,
                         'title': 'Provider Id',
                         'type': 'string'},
                        {'default': True,
                         'description': 'Whether unscoped syncs include this source.',
                         'jsonSchemaType': 'boolean',
                         'logicalType': 'bool',
                         'name': 'enabled',
                         'required': False,
                         'title': 'Enabled',
                         'type': 'boolean'},
                        {'description': 'Provider version metadata.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'version',
                         'required': False,
                         'title': 'Version',
                         'type': 'object'},
                        {'description': 'Source configuration and sync metadata.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'raw_metadata',
                         'required': False,
                         'title': 'Raw Metadata',
                         'type': 'object'},
                        {'description': 'Unknown top-level record fields preserved across storage '
                                        'round trips.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'extra_payload',
                         'required': False,
                         'title': 'Extra Payload',
                         'type': 'object'},
                        {'default': None,
                         'description': 'Last successful source sync timestamp.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'datetime | null',
                         'name': 'synced_at',
                         'required': False,
                         'title': 'Synced At',
                         'type': 'string'}]},
  'title': 'Sources'},
 {'description': 'Durable normalized company or organization hiring boards.',
  'name': 'boards',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable normalized board key.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'key',
                         'required': True,
                         'title': 'Key',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Source key that emitted this board.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'source_key',
                         'required': True,
                         'title': 'Source Key',
                         'type': 'string'},
                        {'description': 'All sources that currently contain this board domain.',
                         'jsonSchemaType': 'array',
                         'logicalType': 'array<str>',
                         'name': 'source_keys',
                         'required': False,
                         'title': 'Source Keys',
                         'type': 'array'},
                        {'description': 'Source-specific emitted board keys merged into this '
                                        'board.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, str>',
                         'name': 'source_board_keys',
                         'required': False,
                         'title': 'Source Board Keys',
                         'type': 'object'},
                        {'constraints': {'required': True},
                         'description': 'Provider-native board identifier.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'remote_id',
                         'required': True,
                         'title': 'Remote Id',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Provider-native slug.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'remote_slug',
                         'required': False,
                         'title': 'Remote Slug',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Company or board display name.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'name',
                         'required': True,
                         'title': 'Name',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Normalized website domain.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'domain',
                         'required': False,
                         'title': 'Domain',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Canonical company website URL.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'website_url',
                         'required': False,
                         'title': 'Website Url',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Provider-supplied board description.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'description',
                         'required': False,
                         'title': 'Description',
                         'type': 'string'},
                        {'description': 'Industry or market tags.',
                         'jsonSchemaType': 'array',
                         'logicalType': 'array<str>',
                         'name': 'markets',
                         'required': False,
                         'title': 'Markets',
                         'type': 'array'},
                        {'description': 'Office or hiring locations.',
                         'jsonSchemaType': 'array',
                         'logicalType': 'array<str>',
                         'name': 'locations',
                         'required': False,
                         'title': 'Locations',
                         'type': 'array'},
                        {'default': None,
                         'description': 'Employee or team-size estimate.',
                         'jsonSchemaType': 'integer | null',
                         'logicalType': 'int | null',
                         'name': 'staff_count',
                         'required': False,
                         'title': 'Staff Count',
                         'type': 'integer'},
                        {'default': None,
                         'description': 'Approximate number of open jobs.',
                         'jsonSchemaType': 'integer | null',
                         'logicalType': 'int | null',
                         'name': 'num_jobs_hint',
                         'required': False,
                         'title': 'Num Jobs Hint',
                         'type': 'integer'},
                        {'description': 'Unmodified upstream board payload.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'raw_payload',
                         'required': False,
                         'title': 'Raw Payload',
                         'type': 'object'},
                        {'description': 'Unknown top-level record fields preserved across storage '
                                        'round trips.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'extra_payload',
                         'required': False,
                         'title': 'Extra Payload',
                         'type': 'object'},
                        {'default': None,
                         'description': 'Last successful board sync timestamp.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'datetime | null',
                         'name': 'synced_at',
                         'required': False,
                         'title': 'Synced At',
                         'type': 'string'}]},
  'title': 'Boards'},
 {'description': 'Durable provider routes that connect boards to upstream systems.',
  'name': 'board_providers',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable route primary key.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Source key that reported this route.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'source_key',
                         'required': True,
                         'title': 'Source Key',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Board key this route belongs to.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'board_key',
                         'required': True,
                         'title': 'Board Key',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Provider adapter identifier.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'provider_id',
                         'required': True,
                         'title': 'Provider Id',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Human-readable upstream route label.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'label',
                         'required': False,
                         'title': 'Label',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Normalized provider support level.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'support_level',
                         'required': True,
                         'title': 'Support Level',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Approximate provider-reported job count.',
                         'jsonSchemaType': 'integer | null',
                         'logicalType': 'int | null',
                         'name': 'count_hint',
                         'required': False,
                         'title': 'Count Hint',
                         'type': 'integer'},
                        {'default': None,
                         'description': 'Hosted job board URL.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'board_url',
                         'required': False,
                         'title': 'Board Url',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Provider-specific board token or slug.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'token',
                         'required': False,
                         'title': 'Token',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Multi-tenant provider host.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'host',
                         'required': False,
                         'title': 'Host',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Multi-tenant provider tenant.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'tenant',
                         'required': False,
                         'title': 'Tenant',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Multi-tenant provider site path.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'site',
                         'required': False,
                         'title': 'Site',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Last probe or sync status.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'last_status',
                         'required': False,
                         'title': 'Last Status',
                         'type': 'string'},
                        {'description': 'Unmodified upstream route payload.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'raw_payload',
                         'required': False,
                         'title': 'Raw Payload',
                         'type': 'object'},
                        {'description': 'Unknown top-level record fields preserved across storage '
                                        'round trips.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'extra_payload',
                         'required': False,
                         'title': 'Extra Payload',
                         'type': 'object'},
                        {'default': None,
                         'description': 'Route discovery timestamp.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'datetime | null',
                         'name': 'detected_at',
                         'required': False,
                         'title': 'Detected At',
                         'type': 'string'}]},
  'title': 'Board Providers'},
 {'description': 'Stable job identities and lifecycle state.',
  'name': 'jobs',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable normalized job identity.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Board key this job belongs to.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'board_key',
                         'required': True,
                         'title': 'Board Key',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Provider adapter identifier.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'provider_id',
                         'required': True,
                         'title': 'Provider Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Provider-native job identifier.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'remote_id',
                         'required': True,
                         'title': 'Remote Id',
                         'type': 'string'},
                        {'default': 'open',
                         'description': 'Current lifecycle status for the stable job identity.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'status',
                         'required': False,
                         'title': 'Status',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Current normalized job version id.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'current_version_id',
                         'required': False,
                         'title': 'Current Version Id',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Current normalized content hash.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'current_content_hash',
                         'required': False,
                         'title': 'Current Content Hash',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Current raw payload-pair hash.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'current_payload_hash',
                         'required': False,
                         'title': 'Current Payload Hash',
                         'type': 'string'},
                        {'description': 'First successful route sync that observed this job '
                                        'identity.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'first_seen_at',
                         'required': False,
                         'title': 'First Seen At',
                         'type': 'datetime'},
                        {'description': 'Most recent successful route sync that observed this job '
                                        'identity.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'last_seen_at',
                         'required': False,
                         'title': 'Last Seen At',
                         'type': 'datetime'},
                        {'default': None,
                         'description': 'Route sync timestamp when this job disappeared while '
                                        'open.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'datetime | null',
                         'name': 'closed_at',
                         'required': False,
                         'title': 'Closed At',
                         'type': 'string'},
                        {'description': 'Last successful lifecycle update timestamp.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'synced_at',
                         'required': False,
                         'title': 'Synced At',
                         'type': 'datetime'},
                        {'description': 'Unknown top-level identity fields preserved across '
                                        'storage round trips.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'extra_payload',
                         'required': False,
                         'title': 'Extra Payload',
                         'type': 'object'}]},
  'title': 'Jobs'},
 {'description': 'Versioned normalized job content snapshots.',
  'name': 'job_versions',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Job version id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Stable job id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'job_id',
                         'required': True,
                         'title': 'Job Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Monotonic version number.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'version',
                         'required': True,
                         'title': 'Version',
                         'type': 'integer'},
                        {'constraints': {'required': True},
                         'description': 'Normalized user-visible content hash.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'content_hash',
                         'required': True,
                         'title': 'Content Hash',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Raw payload-pair hash observed for version.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'payload_hash',
                         'required': True,
                         'title': 'Payload Hash',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Public job title.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'title',
                         'required': True,
                         'title': 'Title',
                         'type': 'string'},
                        {'description': 'Provider-reported job locations.',
                         'jsonSchemaType': 'array',
                         'logicalType': 'array<str>',
                         'name': 'locations',
                         'required': False,
                         'title': 'Locations',
                         'type': 'array'},
                        {'default': None,
                         'description': 'Provider-reported department.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'department',
                         'required': False,
                         'title': 'Department',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Provider-reported team or group.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'team',
                         'required': False,
                         'title': 'Team',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Workplace, commitment, or time-type label.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'workplace_type',
                         'required': False,
                         'title': 'Workplace Type',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Board or company display name.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'company',
                         'required': False,
                         'title': 'Company',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Employment or commitment type.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'employment_type',
                         'required': False,
                         'title': 'Employment Type',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Plain-text provider job description.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'description',
                         'required': False,
                         'title': 'Description',
                         'type': 'string'},
                        {'default': None,
                         'description': 'HTML provider job description.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'description_html',
                         'required': False,
                         'title': 'Description Html',
                         'type': 'string'},
                        {'default': None,
                         'description': 'JSON Resume-compatible remote work level.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'remote',
                         'required': False,
                         'title': 'Remote',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Provider compensation payload or normalized compensation '
                                        'details.',
                         'jsonSchemaType': 'object | null',
                         'logicalType': 'object<str, JsonValue> | null',
                         'name': 'compensation',
                         'required': False,
                         'title': 'Compensation',
                         'type': 'object'},
                        {'default': None,
                         'description': 'JSON Resume-compatible salary display string.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'salary',
                         'required': False,
                         'title': 'Salary',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Minimum deterministic salary or compensation value.',
                         'jsonSchemaType': 'number | null',
                         'logicalType': 'float | null',
                         'name': 'salary_min',
                         'required': False,
                         'title': 'Salary Min',
                         'type': 'number'},
                        {'default': None,
                         'description': 'Maximum deterministic salary or compensation value.',
                         'jsonSchemaType': 'number | null',
                         'logicalType': 'float | null',
                         'name': 'salary_max',
                         'required': False,
                         'title': 'Salary Max',
                         'type': 'number'},
                        {'default': None,
                         'description': 'Salary or compensation currency code.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'salary_currency',
                         'required': False,
                         'title': 'Salary Currency',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Experience label when deterministically available.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'experience',
                         'required': False,
                         'title': 'Experience',
                         'type': 'string'},
                        {'description': 'Deterministic responsibility bullets.',
                         'jsonSchemaType': 'array',
                         'logicalType': 'array<str>',
                         'name': 'responsibilities',
                         'required': False,
                         'title': 'Responsibilities',
                         'type': 'array'},
                        {'description': 'Deterministic qualification bullets.',
                         'jsonSchemaType': 'array',
                         'logicalType': 'array<str>',
                         'name': 'qualifications',
                         'required': False,
                         'title': 'Qualifications',
                         'type': 'array'},
                        {'description': 'JSON Resume-compatible skill objects.',
                         'jsonSchemaType': 'array',
                         'logicalType': 'array<object<str, JsonValue>>',
                         'name': 'skills',
                         'required': False,
                         'title': 'Skills',
                         'type': 'array'},
                        {'default': None,
                         'description': 'JSON Resume-compatible job-description object.',
                         'jsonSchemaType': 'object | null',
                         'logicalType': 'object<str, JsonValue> | null',
                         'name': 'job_description',
                         'required': False,
                         'title': 'Job Description',
                         'type': 'object'},
                        {'default': None,
                         'description': 'Canonical public posting URL.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'posting_url',
                         'required': False,
                         'title': 'Posting Url',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Direct application URL.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'apply_url',
                         'required': False,
                         'title': 'Apply Url',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Provider-native posted timestamp.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'posted_at',
                         'required': False,
                         'title': 'Posted At',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Provider-native updated timestamp.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'updated_at',
                         'required': False,
                         'title': 'Updated At',
                         'type': 'string'},
                        {'description': 'Unknown top-level version fields preserved across storage '
                                        'round trips.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'extra_payload',
                         'required': False,
                         'title': 'Extra Payload',
                         'type': 'object'},
                        {'description': 'First sync timestamp that observed this normalized '
                                        'content.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'first_seen_at',
                         'required': False,
                         'title': 'First Seen At',
                         'type': 'datetime'},
                        {'description': 'Most recent sync timestamp that observed this normalized '
                                        'content.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'last_seen_at',
                         'required': False,
                         'title': 'Last Seen At',
                         'type': 'datetime'},
                        {'description': 'UTC timestamp when this version row was created.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'created_at',
                         'required': False,
                         'title': 'Created At',
                         'type': 'datetime'}]},
  'title': 'Job Versions'},
 {'description': 'Indexed location labels for each normalized job version.',
  'name': 'job_version_locations',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable job-version location row id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Job version this location belongs to.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'job_version_id',
                         'required': True,
                         'title': 'Job Version Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Zero-based location order within the job version.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'ordinal',
                         'required': True,
                         'title': 'Ordinal',
                         'type': 'integer'},
                        {'constraints': {'required': True},
                         'description': 'Location label text.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'label',
                         'required': True,
                         'title': 'Label',
                         'type': 'string'}]},
  'title': 'Job Version Locations'},
 {'description': 'Indexed skill groups for each normalized job version.',
  'name': 'job_version_skills',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable job-version skill row id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Job version this skill group belongs to.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'job_version_id',
                         'required': True,
                         'title': 'Job Version Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Zero-based skill order within the job version.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'ordinal',
                         'required': True,
                         'title': 'Ordinal',
                         'type': 'integer'},
                        {'default': None,
                         'description': 'Skill group display name.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'name',
                         'required': False,
                         'title': 'Name',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Skill group proficiency or level label.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'level',
                         'required': False,
                         'title': 'Level',
                         'type': 'string'}]},
  'title': 'Job Version Skills'},
 {'description': 'Indexed skill keywords for each normalized job version skill.',
  'name': 'job_version_skill_keywords',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable job-version skill keyword row id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Skill group this keyword belongs to.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'skill_id',
                         'required': True,
                         'title': 'Skill Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Zero-based keyword order within the skill group.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'ordinal',
                         'required': True,
                         'title': 'Ordinal',
                         'type': 'integer'},
                        {'constraints': {'required': True},
                         'description': 'Skill keyword text.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'keyword',
                         'required': True,
                         'title': 'Keyword',
                         'type': 'string'}]},
  'title': 'Job Version Skill Keywords'},
 {'description': 'Indexed responsibility and qualification bullets for each job version.',
  'name': 'job_version_bullets',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable job-version bullet row id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Job version this bullet belongs to.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'job_version_id',
                         'required': True,
                         'title': 'Job Version Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Bullet category, such as responsibility or qualification.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'kind',
                         'required': True,
                         'title': 'Kind',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Zero-based bullet order within its category.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'ordinal',
                         'required': True,
                         'title': 'Ordinal',
                         'type': 'integer'},
                        {'constraints': {'required': True},
                         'description': 'Bullet text.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'text',
                         'required': True,
                         'title': 'Text',
                         'type': 'string'}]},
  'title': 'Job Version Bullets'},
 {'description': 'Raw upstream payload snapshots for audit and replay.',
  'name': 'job_payload_snapshots',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable raw payload snapshot row id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Stable job identity this raw payload belongs to.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'job_id',
                         'required': True,
                         'title': 'Job Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Raw payload source kind, such as listing or detail.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'payload_kind',
                         'required': True,
                         'title': 'Payload Kind',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Canonical hash of the unmodified raw payload.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'payload_hash',
                         'required': True,
                         'title': 'Payload Hash',
                         'type': 'string'},
                        {'description': 'Unmodified upstream payload for audit and replay.',
                         'jsonSchemaType': 'object',
                         'logicalType': 'object<str, JsonValue>',
                         'name': 'payload',
                         'required': False,
                         'title': 'Payload',
                         'type': 'object'},
                        {'description': 'UTC sync timestamp when this raw payload was observed.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'observed_at',
                         'required': False,
                         'title': 'Observed At',
                         'type': 'datetime'}]},
  'title': 'Job Payload Snapshots'},
 {'description': 'Provider route sync attempts and aggregate change counts.',
  'name': 'job_sync_runs',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable provider route sync run id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Board key synced during this provider route run.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'board_key',
                         'required': True,
                         'title': 'Board Key',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Provider route synced during this run.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'provider_id',
                         'required': True,
                         'title': 'Provider Id',
                         'type': 'string'},
                        {'description': 'UTC timestamp for this provider route sync attempt.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'synced_at',
                         'required': False,
                         'title': 'Synced At',
                         'type': 'datetime'},
                        {'default': True,
                         'description': 'Whether the route sync completed.',
                         'jsonSchemaType': 'boolean',
                         'logicalType': 'bool',
                         'name': 'success',
                         'required': False,
                         'title': 'Success',
                         'type': 'boolean'},
                        {'default': None,
                         'description': 'Error message captured for failed route syncs.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'error',
                         'required': False,
                         'title': 'Error',
                         'type': 'string'},
                        {'default': 0,
                         'description': 'Total jobs observed in the route sync.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'job_count',
                         'required': False,
                         'title': 'Job Count',
                         'type': 'integer'},
                        {'default': 0,
                         'description': 'Jobs newly created by the route sync.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'new_count',
                         'required': False,
                         'title': 'New Count',
                         'type': 'integer'},
                        {'default': 0,
                         'description': 'Jobs observed without content changes.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'unchanged_count',
                         'required': False,
                         'title': 'Unchanged Count',
                         'type': 'integer'},
                        {'default': 0,
                         'description': 'Jobs with a new normalized content version.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'changed_count',
                         'required': False,
                         'title': 'Changed Count',
                         'type': 'integer'},
                        {'default': 0,
                         'description': 'Previously closed jobs reopened by the route sync.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'reopened_count',
                         'required': False,
                         'title': 'Reopened Count',
                         'type': 'integer'},
                        {'default': 0,
                         'description': 'Previously open jobs closed by the route sync.',
                         'jsonSchemaType': 'integer',
                         'logicalType': 'int',
                         'name': 'closed_count',
                         'required': False,
                         'title': 'Closed Count',
                         'type': 'integer'}]},
  'title': 'Job Sync Runs'},
 {'description': 'Per-job observations recorded during provider route syncs.',
  'name': 'job_sync_observations',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'Stable sync observation row id.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'id',
                         'required': True,
                         'title': 'Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Route sync run that recorded this observation.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'sync_run_id',
                         'required': True,
                         'title': 'Sync Run Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Stable job identity observed during sync.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'job_id',
                         'required': True,
                         'title': 'Job Id',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Normalized job version associated with this observation.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'job_version_id',
                         'required': False,
                         'title': 'Job Version Id',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Observation category, such as new, unchanged, changed, '
                                        'reopened, or closed.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'observation_kind',
                         'required': True,
                         'title': 'Observation Kind',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Normalized content hash observed during sync.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'content_hash',
                         'required': False,
                         'title': 'Content Hash',
                         'type': 'string'},
                        {'default': None,
                         'description': 'Raw payload-pair hash observed during sync.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'payload_hash',
                         'required': False,
                         'title': 'Payload Hash',
                         'type': 'string'},
                        {'description': 'UTC timestamp when the observation was recorded.',
                         'format': 'date-time',
                         'jsonSchemaType': 'string',
                         'logicalType': 'datetime',
                         'name': 'observed_at',
                         'required': False,
                         'title': 'Observed At',
                         'type': 'datetime'}]},
  'title': 'Job Sync Observations'},
 {'description': 'In-database table labels and descriptions for openoppsdb.sqlite.',
  'name': 'openopps_tables',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'SQLite table name.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'table_name',
                         'required': True,
                         'title': 'Table Name',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Human-readable table label.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'table_title',
                         'required': True,
                         'title': 'Table Title',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Plain-language table description.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'table_description',
                         'required': True,
                         'title': 'Table Description',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'CSV export path for this SQLite table.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'csv_path',
                         'required': True,
                         'title': 'Csv Path',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Parquet export path for this SQLite table.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'parquet_path',
                         'required': True,
                         'title': 'Parquet Path',
                         'type': 'string'}]},
  'title': 'Openopps Tables'},
 {'description': 'In-database column labels, descriptions, and schema hints for openoppsdb.sqlite.',
  'name': 'openopps_columns',
  'schema': {'fields': [{'constraints': {'required': True},
                         'description': 'SQLite table that owns this column.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'table_name',
                         'required': True,
                         'title': 'Table Name',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'SQLite column name.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'column_name',
                         'required': True,
                         'title': 'Column Name',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Human-readable column label.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'column_title',
                         'required': True,
                         'title': 'Column Title',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Plain-language column description.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'column_description',
                         'required': True,
                         'title': 'Column Description',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Python or typing-level logical type label.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'logical_type',
                         'required': True,
                         'title': 'Logical Type',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'JSON Schema type derived from the model field.',
                         'jsonSchemaType': 'string',
                         'logicalType': 'str',
                         'name': 'json_schema_type',
                         'required': True,
                         'title': 'Json Schema Type',
                         'type': 'string'},
                        {'constraints': {'required': True},
                         'description': 'Whether the source model marks the column as required.',
                         'jsonSchemaType': 'boolean',
                         'logicalType': 'bool',
                         'name': 'required',
                         'required': True,
                         'title': 'Required',
                         'type': 'boolean'},
                        {'default': None,
                         'description': 'Original source alias when it differs from the column '
                                        'name.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'source_name',
                         'required': False,
                         'title': 'Source Name',
                         'type': 'string'},
                        {'default': None,
                         'description': 'JSON Schema format hint, when available.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'format',
                         'required': False,
                         'title': 'Format',
                         'type': 'string'},
                        {'default': None,
                         'description': 'JSON array of allowed values, when available.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'enum_json',
                         'required': False,
                         'title': 'Enum Json',
                         'type': 'string'},
                        {'default': None,
                         'description': 'JSON array of example values, when available.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'examples_json',
                         'required': False,
                         'title': 'Examples Json',
                         'type': 'string'},
                        {'default': None,
                         'description': 'JSON-encoded default value, when available.',
                         'jsonSchemaType': 'string | null',
                         'logicalType': 'str | null',
                         'name': 'default_json',
                         'required': False,
                         'title': 'Default Json',
                         'type': 'string'}]},
  'title': 'Openopps Columns'}]
OPENOPPS_TABLE_ROWS = [{'csv_path': 'exports/csv/sources.csv',
  'parquet_path': 'exports/parquet/sources.parquet',
  'table_description': 'Durable source catalogs that discover company boards.',
  'table_name': 'sources',
  'table_title': 'Sources'},
 {'csv_path': 'exports/csv/boards.csv',
  'parquet_path': 'exports/parquet/boards.parquet',
  'table_description': 'Durable normalized company or organization hiring boards.',
  'table_name': 'boards',
  'table_title': 'Boards'},
 {'csv_path': 'exports/csv/board_providers.csv',
  'parquet_path': 'exports/parquet/board_providers.parquet',
  'table_description': 'Durable provider routes that connect boards to upstream systems.',
  'table_name': 'board_providers',
  'table_title': 'Board Providers'},
 {'csv_path': 'exports/csv/jobs.csv',
  'parquet_path': 'exports/parquet/jobs.parquet',
  'table_description': 'Stable job identities and lifecycle state.',
  'table_name': 'jobs',
  'table_title': 'Jobs'},
 {'csv_path': 'exports/csv/job_versions.csv',
  'parquet_path': 'exports/parquet/job_versions.parquet',
  'table_description': 'Versioned normalized job content snapshots.',
  'table_name': 'job_versions',
  'table_title': 'Job Versions'},
 {'csv_path': 'exports/csv/job_version_locations.csv',
  'parquet_path': 'exports/parquet/job_version_locations.parquet',
  'table_description': 'Indexed location labels for each normalized job version.',
  'table_name': 'job_version_locations',
  'table_title': 'Job Version Locations'},
 {'csv_path': 'exports/csv/job_version_skills.csv',
  'parquet_path': 'exports/parquet/job_version_skills.parquet',
  'table_description': 'Indexed skill groups for each normalized job version.',
  'table_name': 'job_version_skills',
  'table_title': 'Job Version Skills'},
 {'csv_path': 'exports/csv/job_version_skill_keywords.csv',
  'parquet_path': 'exports/parquet/job_version_skill_keywords.parquet',
  'table_description': 'Indexed skill keywords for each normalized job version skill.',
  'table_name': 'job_version_skill_keywords',
  'table_title': 'Job Version Skill Keywords'},
 {'csv_path': 'exports/csv/job_version_bullets.csv',
  'parquet_path': 'exports/parquet/job_version_bullets.parquet',
  'table_description': 'Indexed responsibility and qualification bullets for each job version.',
  'table_name': 'job_version_bullets',
  'table_title': 'Job Version Bullets'},
 {'csv_path': 'exports/csv/job_payload_snapshots.csv',
  'parquet_path': 'exports/parquet/job_payload_snapshots.parquet',
  'table_description': 'Raw upstream payload snapshots for audit and replay.',
  'table_name': 'job_payload_snapshots',
  'table_title': 'Job Payload Snapshots'},
 {'csv_path': 'exports/csv/job_sync_runs.csv',
  'parquet_path': 'exports/parquet/job_sync_runs.parquet',
  'table_description': 'Provider route sync attempts and aggregate change counts.',
  'table_name': 'job_sync_runs',
  'table_title': 'Job Sync Runs'},
 {'csv_path': 'exports/csv/job_sync_observations.csv',
  'parquet_path': 'exports/parquet/job_sync_observations.parquet',
  'table_description': 'Per-job observations recorded during provider route syncs.',
  'table_name': 'job_sync_observations',
  'table_title': 'Job Sync Observations'},
 {'csv_path': 'exports/csv/openopps_tables.csv',
  'parquet_path': 'exports/parquet/openopps_tables.parquet',
  'table_description': 'In-database table labels and descriptions for openoppsdb.sqlite.',
  'table_name': 'openopps_tables',
  'table_title': 'Openopps Tables'},
 {'csv_path': 'exports/csv/openopps_columns.csv',
  'parquet_path': 'exports/parquet/openopps_columns.parquet',
  'table_description': 'In-database column labels, descriptions, and schema hints for '
                       'openoppsdb.sqlite.',
  'table_name': 'openopps_columns',
  'table_title': 'Openopps Columns'}]
OPENOPPS_COLUMN_ROWS = [{'column_description': 'Stable local source key.',
  'column_name': 'key',
  'column_title': 'Key',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Canonical source URL or synthetic manual source URI.',
  'column_name': 'url',
  'column_title': 'Url',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Source adapter identifier.',
  'column_name': 'provider_id',
  'column_title': 'Provider Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Whether unscoped syncs include this source.',
  'column_name': 'enabled',
  'column_title': 'Enabled',
  'default_json': 'true',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'boolean',
  'logical_type': 'bool',
  'required': 0,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Provider version metadata.',
  'column_name': 'version',
  'column_title': 'Version',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Source configuration and sync metadata.',
  'column_name': 'raw_metadata',
  'column_title': 'Raw Metadata',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Unknown top-level record fields preserved across storage round trips.',
  'column_name': 'extra_payload',
  'column_title': 'Extra Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Last successful source sync timestamp.',
  'column_name': 'synced_at',
  'column_title': 'Synced At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'datetime | null',
  'required': 0,
  'source_name': None,
  'table_name': 'sources'},
 {'column_description': 'Stable normalized board key.',
  'column_name': 'key',
  'column_title': 'Key',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Source key that emitted this board.',
  'column_name': 'source_key',
  'column_title': 'Source Key',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'All sources that currently contain this board domain.',
  'column_name': 'source_keys',
  'column_title': 'Source Keys',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'array',
  'logical_type': 'array<str>',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Source-specific emitted board keys merged into this board.',
  'column_name': 'source_board_keys',
  'column_title': 'Source Board Keys',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, str>',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Provider-native board identifier.',
  'column_name': 'remote_id',
  'column_title': 'Remote Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Provider-native slug.',
  'column_name': 'remote_slug',
  'column_title': 'Remote Slug',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Company or board display name.',
  'column_name': 'name',
  'column_title': 'Name',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Normalized website domain.',
  'column_name': 'domain',
  'column_title': 'Domain',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Canonical company website URL.',
  'column_name': 'website_url',
  'column_title': 'Website Url',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Provider-supplied board description.',
  'column_name': 'description',
  'column_title': 'Description',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Industry or market tags.',
  'column_name': 'markets',
  'column_title': 'Markets',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'array',
  'logical_type': 'array<str>',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Office or hiring locations.',
  'column_name': 'locations',
  'column_title': 'Locations',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'array',
  'logical_type': 'array<str>',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Employee or team-size estimate.',
  'column_name': 'staff_count',
  'column_title': 'Staff Count',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer | null',
  'logical_type': 'int | null',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Approximate number of open jobs.',
  'column_name': 'num_jobs_hint',
  'column_title': 'Num Jobs Hint',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer | null',
  'logical_type': 'int | null',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Unmodified upstream board payload.',
  'column_name': 'raw_payload',
  'column_title': 'Raw Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Unknown top-level record fields preserved across storage round trips.',
  'column_name': 'extra_payload',
  'column_title': 'Extra Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Last successful board sync timestamp.',
  'column_name': 'synced_at',
  'column_title': 'Synced At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'datetime | null',
  'required': 0,
  'source_name': None,
  'table_name': 'boards'},
 {'column_description': 'Stable route primary key.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Source key that reported this route.',
  'column_name': 'source_key',
  'column_title': 'Source Key',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Board key this route belongs to.',
  'column_name': 'board_key',
  'column_title': 'Board Key',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Provider adapter identifier.',
  'column_name': 'provider_id',
  'column_title': 'Provider Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Human-readable upstream route label.',
  'column_name': 'label',
  'column_title': 'Label',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Normalized provider support level.',
  'column_name': 'support_level',
  'column_title': 'Support Level',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Approximate provider-reported job count.',
  'column_name': 'count_hint',
  'column_title': 'Count Hint',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer | null',
  'logical_type': 'int | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Hosted job board URL.',
  'column_name': 'board_url',
  'column_title': 'Board Url',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Provider-specific board token or slug.',
  'column_name': 'token',
  'column_title': 'Token',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Multi-tenant provider host.',
  'column_name': 'host',
  'column_title': 'Host',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Multi-tenant provider tenant.',
  'column_name': 'tenant',
  'column_title': 'Tenant',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Multi-tenant provider site path.',
  'column_name': 'site',
  'column_title': 'Site',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Last probe or sync status.',
  'column_name': 'last_status',
  'column_title': 'Last Status',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Unmodified upstream route payload.',
  'column_name': 'raw_payload',
  'column_title': 'Raw Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Unknown top-level record fields preserved across storage round trips.',
  'column_name': 'extra_payload',
  'column_title': 'Extra Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Route discovery timestamp.',
  'column_name': 'detected_at',
  'column_title': 'Detected At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'datetime | null',
  'required': 0,
  'source_name': None,
  'table_name': 'board_providers'},
 {'column_description': 'Stable normalized job identity.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Board key this job belongs to.',
  'column_name': 'board_key',
  'column_title': 'Board Key',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Provider adapter identifier.',
  'column_name': 'provider_id',
  'column_title': 'Provider Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Provider-native job identifier.',
  'column_name': 'remote_id',
  'column_title': 'Remote Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Current lifecycle status for the stable job identity.',
  'column_name': 'status',
  'column_title': 'Status',
  'default_json': '"open"',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Current normalized job version id.',
  'column_name': 'current_version_id',
  'column_title': 'Current Version Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Current normalized content hash.',
  'column_name': 'current_content_hash',
  'column_title': 'Current Content Hash',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Current raw payload-pair hash.',
  'column_name': 'current_payload_hash',
  'column_title': 'Current Payload Hash',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'First successful route sync that observed this job identity.',
  'column_name': 'first_seen_at',
  'column_title': 'First Seen At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Most recent successful route sync that observed this job identity.',
  'column_name': 'last_seen_at',
  'column_title': 'Last Seen At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Route sync timestamp when this job disappeared while open.',
  'column_name': 'closed_at',
  'column_title': 'Closed At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'datetime | null',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Last successful lifecycle update timestamp.',
  'column_name': 'synced_at',
  'column_title': 'Synced At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Unknown top-level identity fields preserved across storage round trips.',
  'column_name': 'extra_payload',
  'column_title': 'Extra Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'jobs'},
 {'column_description': 'Job version id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Stable job id.',
  'column_name': 'job_id',
  'column_title': 'Job Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Monotonic version number.',
  'column_name': 'version',
  'column_title': 'Version',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 1,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Normalized user-visible content hash.',
  'column_name': 'content_hash',
  'column_title': 'Content Hash',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Raw payload-pair hash observed for version.',
  'column_name': 'payload_hash',
  'column_title': 'Payload Hash',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Public job title.',
  'column_name': 'title',
  'column_title': 'Title',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Provider-reported job locations.',
  'column_name': 'locations',
  'column_title': 'Locations',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'array',
  'logical_type': 'array<str>',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Provider-reported department.',
  'column_name': 'department',
  'column_title': 'Department',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Provider-reported team or group.',
  'column_name': 'team',
  'column_title': 'Team',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Workplace, commitment, or time-type label.',
  'column_name': 'workplace_type',
  'column_title': 'Workplace Type',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Board or company display name.',
  'column_name': 'company',
  'column_title': 'Company',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Employment or commitment type.',
  'column_name': 'employment_type',
  'column_title': 'Employment Type',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Plain-text provider job description.',
  'column_name': 'description',
  'column_title': 'Description',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'HTML provider job description.',
  'column_name': 'description_html',
  'column_title': 'Description Html',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'JSON Resume-compatible remote work level.',
  'column_name': 'remote',
  'column_title': 'Remote',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Provider compensation payload or normalized compensation details.',
  'column_name': 'compensation',
  'column_title': 'Compensation',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object | null',
  'logical_type': 'object<str, JsonValue> | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'JSON Resume-compatible salary display string.',
  'column_name': 'salary',
  'column_title': 'Salary',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Minimum deterministic salary or compensation value.',
  'column_name': 'salary_min',
  'column_title': 'Salary Min',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'number | null',
  'logical_type': 'float | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Maximum deterministic salary or compensation value.',
  'column_name': 'salary_max',
  'column_title': 'Salary Max',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'number | null',
  'logical_type': 'float | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Salary or compensation currency code.',
  'column_name': 'salary_currency',
  'column_title': 'Salary Currency',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Experience label when deterministically available.',
  'column_name': 'experience',
  'column_title': 'Experience',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Deterministic responsibility bullets.',
  'column_name': 'responsibilities',
  'column_title': 'Responsibilities',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'array',
  'logical_type': 'array<str>',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Deterministic qualification bullets.',
  'column_name': 'qualifications',
  'column_title': 'Qualifications',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'array',
  'logical_type': 'array<str>',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'JSON Resume-compatible skill objects.',
  'column_name': 'skills',
  'column_title': 'Skills',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'array',
  'logical_type': 'array<object<str, JsonValue>>',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'JSON Resume-compatible job-description object.',
  'column_name': 'job_description',
  'column_title': 'Job Description',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object | null',
  'logical_type': 'object<str, JsonValue> | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Canonical public posting URL.',
  'column_name': 'posting_url',
  'column_title': 'Posting Url',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Direct application URL.',
  'column_name': 'apply_url',
  'column_title': 'Apply Url',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Provider-native posted timestamp.',
  'column_name': 'posted_at',
  'column_title': 'Posted At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Provider-native updated timestamp.',
  'column_name': 'updated_at',
  'column_title': 'Updated At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Unknown top-level version fields preserved across storage round trips.',
  'column_name': 'extra_payload',
  'column_title': 'Extra Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'First sync timestamp that observed this normalized content.',
  'column_name': 'first_seen_at',
  'column_title': 'First Seen At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Most recent sync timestamp that observed this normalized content.',
  'column_name': 'last_seen_at',
  'column_title': 'Last Seen At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'UTC timestamp when this version row was created.',
  'column_name': 'created_at',
  'column_title': 'Created At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'job_versions'},
 {'column_description': 'Stable job-version location row id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_locations'},
 {'column_description': 'Job version this location belongs to.',
  'column_name': 'job_version_id',
  'column_title': 'Job Version Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_locations'},
 {'column_description': 'Zero-based location order within the job version.',
  'column_name': 'ordinal',
  'column_title': 'Ordinal',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_locations'},
 {'column_description': 'Location label text.',
  'column_name': 'label',
  'column_title': 'Label',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_locations'},
 {'column_description': 'Stable job-version skill row id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_skills'},
 {'column_description': 'Job version this skill group belongs to.',
  'column_name': 'job_version_id',
  'column_title': 'Job Version Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_skills'},
 {'column_description': 'Zero-based skill order within the job version.',
  'column_name': 'ordinal',
  'column_title': 'Ordinal',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_skills'},
 {'column_description': 'Skill group display name.',
  'column_name': 'name',
  'column_title': 'Name',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_version_skills'},
 {'column_description': 'Skill group proficiency or level label.',
  'column_name': 'level',
  'column_title': 'Level',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_version_skills'},
 {'column_description': 'Stable job-version skill keyword row id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_skill_keywords'},
 {'column_description': 'Skill group this keyword belongs to.',
  'column_name': 'skill_id',
  'column_title': 'Skill Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_skill_keywords'},
 {'column_description': 'Zero-based keyword order within the skill group.',
  'column_name': 'ordinal',
  'column_title': 'Ordinal',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_skill_keywords'},
 {'column_description': 'Skill keyword text.',
  'column_name': 'keyword',
  'column_title': 'Keyword',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_skill_keywords'},
 {'column_description': 'Stable job-version bullet row id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_bullets'},
 {'column_description': 'Job version this bullet belongs to.',
  'column_name': 'job_version_id',
  'column_title': 'Job Version Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_bullets'},
 {'column_description': 'Bullet category, such as responsibility or qualification.',
  'column_name': 'kind',
  'column_title': 'Kind',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_bullets'},
 {'column_description': 'Zero-based bullet order within its category.',
  'column_name': 'ordinal',
  'column_title': 'Ordinal',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_bullets'},
 {'column_description': 'Bullet text.',
  'column_name': 'text',
  'column_title': 'Text',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_version_bullets'},
 {'column_description': 'Stable raw payload snapshot row id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_payload_snapshots'},
 {'column_description': 'Stable job identity this raw payload belongs to.',
  'column_name': 'job_id',
  'column_title': 'Job Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_payload_snapshots'},
 {'column_description': 'Raw payload source kind, such as listing or detail.',
  'column_name': 'payload_kind',
  'column_title': 'Payload Kind',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_payload_snapshots'},
 {'column_description': 'Canonical hash of the unmodified raw payload.',
  'column_name': 'payload_hash',
  'column_title': 'Payload Hash',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_payload_snapshots'},
 {'column_description': 'Unmodified upstream payload for audit and replay.',
  'column_name': 'payload',
  'column_title': 'Payload',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'object',
  'logical_type': 'object<str, JsonValue>',
  'required': 0,
  'source_name': None,
  'table_name': 'job_payload_snapshots'},
 {'column_description': 'UTC sync timestamp when this raw payload was observed.',
  'column_name': 'observed_at',
  'column_title': 'Observed At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'job_payload_snapshots'},
 {'column_description': 'Stable provider route sync run id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Board key synced during this provider route run.',
  'column_name': 'board_key',
  'column_title': 'Board Key',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Provider route synced during this run.',
  'column_name': 'provider_id',
  'column_title': 'Provider Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'UTC timestamp for this provider route sync attempt.',
  'column_name': 'synced_at',
  'column_title': 'Synced At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Whether the route sync completed.',
  'column_name': 'success',
  'column_title': 'Success',
  'default_json': 'true',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'boolean',
  'logical_type': 'bool',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Error message captured for failed route syncs.',
  'column_name': 'error',
  'column_title': 'Error',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Total jobs observed in the route sync.',
  'column_name': 'job_count',
  'column_title': 'Job Count',
  'default_json': '0',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Jobs newly created by the route sync.',
  'column_name': 'new_count',
  'column_title': 'New Count',
  'default_json': '0',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Jobs observed without content changes.',
  'column_name': 'unchanged_count',
  'column_title': 'Unchanged Count',
  'default_json': '0',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Jobs with a new normalized content version.',
  'column_name': 'changed_count',
  'column_title': 'Changed Count',
  'default_json': '0',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Previously closed jobs reopened by the route sync.',
  'column_name': 'reopened_count',
  'column_title': 'Reopened Count',
  'default_json': '0',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Previously open jobs closed by the route sync.',
  'column_name': 'closed_count',
  'column_title': 'Closed Count',
  'default_json': '0',
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'integer',
  'logical_type': 'int',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_runs'},
 {'column_description': 'Stable sync observation row id.',
  'column_name': 'id',
  'column_title': 'Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'Route sync run that recorded this observation.',
  'column_name': 'sync_run_id',
  'column_title': 'Sync Run Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'Stable job identity observed during sync.',
  'column_name': 'job_id',
  'column_title': 'Job Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'Normalized job version associated with this observation.',
  'column_name': 'job_version_id',
  'column_title': 'Job Version Id',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'Observation category, such as new, unchanged, changed, reopened, or '
                        'closed.',
  'column_name': 'observation_kind',
  'column_title': 'Observation Kind',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'Normalized content hash observed during sync.',
  'column_name': 'content_hash',
  'column_title': 'Content Hash',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'Raw payload-pair hash observed during sync.',
  'column_name': 'payload_hash',
  'column_title': 'Payload Hash',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'UTC timestamp when the observation was recorded.',
  'column_name': 'observed_at',
  'column_title': 'Observed At',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': 'date-time',
  'json_schema_type': 'string',
  'logical_type': 'datetime',
  'required': 0,
  'source_name': None,
  'table_name': 'job_sync_observations'},
 {'column_description': 'SQLite table name.',
  'column_name': 'table_name',
  'column_title': 'Table Name',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_tables'},
 {'column_description': 'Human-readable table label.',
  'column_name': 'table_title',
  'column_title': 'Table Title',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_tables'},
 {'column_description': 'Plain-language table description.',
  'column_name': 'table_description',
  'column_title': 'Table Description',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_tables'},
 {'column_description': 'CSV export path for this SQLite table.',
  'column_name': 'csv_path',
  'column_title': 'Csv Path',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_tables'},
 {'column_description': 'Parquet export path for this SQLite table.',
  'column_name': 'parquet_path',
  'column_title': 'Parquet Path',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_tables'},
 {'column_description': 'SQLite table that owns this column.',
  'column_name': 'table_name',
  'column_title': 'Table Name',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'SQLite column name.',
  'column_name': 'column_name',
  'column_title': 'Column Name',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'Human-readable column label.',
  'column_name': 'column_title',
  'column_title': 'Column Title',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'Plain-language column description.',
  'column_name': 'column_description',
  'column_title': 'Column Description',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'Python or typing-level logical type label.',
  'column_name': 'logical_type',
  'column_title': 'Logical Type',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'JSON Schema type derived from the model field.',
  'column_name': 'json_schema_type',
  'column_title': 'Json Schema Type',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string',
  'logical_type': 'str',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'Whether the source model marks the column as required.',
  'column_name': 'required',
  'column_title': 'Required',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'boolean',
  'logical_type': 'bool',
  'required': 1,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'Original source alias when it differs from the column name.',
  'column_name': 'source_name',
  'column_title': 'Source Name',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'JSON Schema format hint, when available.',
  'column_name': 'format',
  'column_title': 'Format',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'JSON array of allowed values, when available.',
  'column_name': 'enum_json',
  'column_title': 'Enum Json',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'JSON array of example values, when available.',
  'column_name': 'examples_json',
  'column_title': 'Examples Json',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'openopps_columns'},
 {'column_description': 'JSON-encoded default value, when available.',
  'column_name': 'default_json',
  'column_title': 'Default Json',
  'default_json': None,
  'enum_json': None,
  'examples_json': None,
  'format': None,
  'json_schema_type': 'string | null',
  'logical_type': 'str | null',
  'required': 0,
  'source_name': None,
  'table_name': 'openopps_columns'}]
PUBLIC_UPLOAD_DATA_FILES = ('openoppsdb.sqlite',
 'exports/csv/sources.csv',
 'exports/csv/boards.csv',
 'exports/csv/board_providers.csv',
 'exports/csv/jobs.csv',
 'exports/csv/job_versions.csv',
 'exports/csv/job_version_locations.csv',
 'exports/csv/job_version_skills.csv',
 'exports/csv/job_version_skill_keywords.csv',
 'exports/csv/job_version_bullets.csv',
 'exports/csv/job_payload_snapshots.csv',
 'exports/csv/job_sync_runs.csv',
 'exports/csv/job_sync_observations.csv',
 'exports/csv/openopps_tables.csv',
 'exports/csv/openopps_columns.csv',
 'exports/parquet/sources.parquet',
 'exports/parquet/boards.parquet',
 'exports/parquet/board_providers.parquet',
 'exports/parquet/jobs.parquet',
 'exports/parquet/job_versions.parquet',
 'exports/parquet/job_version_locations.parquet',
 'exports/parquet/job_version_skills.parquet',
 'exports/parquet/job_version_skill_keywords.parquet',
 'exports/parquet/job_version_bullets.parquet',
 'exports/parquet/job_payload_snapshots.parquet',
 'exports/parquet/job_sync_runs.parquet',
 'exports/parquet/job_sync_observations.parquet',
 'exports/parquet/openopps_tables.parquet',
 'exports/parquet/openopps_columns.parquet')
SKILL_TEXT_VALUE_LIMIT = 4000
SKILL_LEVEL_ALIASES = (
    ("Executive", ("chief", "c-level", "c suite", "vp", "vice president")),
    ("Principal", ("principal", "staff")),
    ("Senior", ("senior", "sr", "lead")),
    ("Manager", ("manager", "director", "head of")),
    ("Junior", ("junior", "jr", "entry level", "intern", "associate")),
)
SLUG_RE = re.compile(r"[^a-z0-9]+")
KAGGLE_SYNC_TIMEOUT_SECONDS = float(
    os.environ.get(
        "OPENOPPS_KAGGLE_SYNC_TIMEOUT_SECONDS",
        "3300",
    )
)
KAGGLE_JOB_ROUTE_LIMIT = int(
    os.environ.get(
        "OPENOPPS_KAGGLE_JOB_ROUTE_LIMIT",
        "120",
    )
)
KAGGLE_CREDENTIALS_ERROR = (
    "Kaggle API credentials are required to publish openoppsdb. "
    "Configure KAGGLE_USERNAME and KAGGLE_KEY as Kaggle notebook secrets "
    "before running the manager."
)
KAGGLE_SECRET_RETRIES = int(os.environ.get("OPENOPPS_KAGGLE_SECRET_RETRIES", "30"))
KAGGLE_SECRET_RETRY_SECONDS = float(
    os.environ.get("OPENOPPS_KAGGLE_SECRET_RETRY_SECONDS", "10")
)
KAGGLE_SECRET_LOOKUP_ERRORS: dict[str, str] = {}

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def normalize_kaggle_notebook_secret(value: object) -> str | None:
    if isinstance(value, str) and value.strip():
        return value.strip()
    return None

def read_kaggle_notebook_secrets() -> tuple[str | None, str | None]:
    last_key = None
    last_username = None
    for attempt in range(1, KAGGLE_SECRET_RETRIES + 1):
        key_error = None
        username_error = None
        try:
            from kaggle_secrets import UserSecretsClient
        except Exception as exc:
            KAGGLE_SECRET_LOOKUP_ERRORS["kaggle_secrets"] = type(exc).__name__
            print(f"Kaggle notebook secrets client unavailable: {type(exc).__name__}")
            return last_key, last_username

        user_secrets = UserSecretsClient()
        try:
            secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
        except Exception as exc:
            key_error = type(exc).__name__
            KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_KEY"] = key_error
            print(
                f"KAGGLE_KEY notebook secret lookup failed "
                f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES}): "
                f"{key_error}"
            )
            secret_value_0 = None

        try:
            secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")
        except Exception as exc:
            username_error = type(exc).__name__
            KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_USERNAME"] = username_error
            print(
                f"KAGGLE_USERNAME notebook secret lookup failed "
                f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES}): "
                f"{username_error}"
            )
            secret_value_1 = None

        key = normalize_kaggle_notebook_secret(secret_value_0)
        username = normalize_kaggle_notebook_secret(secret_value_1)
        if key:
            last_key = key
            KAGGLE_SECRET_LOOKUP_ERRORS.pop("KAGGLE_KEY", None)
        elif key_error is None:
            KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_KEY"] = "NotFound"
            print(
                f"KAGGLE_KEY not found in Kaggle notebook secrets "
                f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES})."
            )

        if username:
            last_username = username
            KAGGLE_SECRET_LOOKUP_ERRORS.pop("KAGGLE_USERNAME", None)
        elif username_error is None:
            KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_USERNAME"] = "NotFound"
            print(
                f"KAGGLE_USERNAME not found in Kaggle notebook secrets "
                f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES})."
            )

        if key and username:
            return key, username

        if attempt < KAGGLE_SECRET_RETRIES:
            time.sleep(KAGGLE_SECRET_RETRY_SECONDS)
            continue
    return last_key, last_username

def load_kaggle_notebook_secrets() -> None:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        print("KAGGLE_USERNAME and KAGGLE_KEY already present in environment.")
        return

    key, username = read_kaggle_notebook_secrets()

    if os.environ.get("KAGGLE_USERNAME"):
        print("KAGGLE_USERNAME already present in environment.")
    elif username:
        os.environ["KAGGLE_USERNAME"] = username
        print("KAGGLE_USERNAME loaded from Kaggle notebook secrets.")

    if os.environ.get("KAGGLE_KEY"):
        print("KAGGLE_KEY already present in environment.")
    elif key:
        os.environ["KAGGLE_KEY"] = key
        print("KAGGLE_KEY loaded from Kaggle notebook secrets.")

def has_kaggle_credentials() -> bool:
    load_kaggle_notebook_secrets()
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    token_path = os.environ.get("KAGGLE_API_V1_TOKEN_PATH")
    return bool(
        os.environ.get("KAGGLE_API_TOKEN")
        or (token_path and Path(token_path).expanduser().exists())
        or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
        or kaggle_json.exists()
    )

def require_kaggle_credentials() -> None:
    if not has_kaggle_credentials():
        if KAGGLE_SECRET_LOOKUP_ERRORS:
            details = ", ".join(
                f"{key}={value}"
                for key, value in sorted(KAGGLE_SECRET_LOOKUP_ERRORS.items())
            )
            raise RuntimeError(f"{KAGGLE_CREDENTIALS_ERROR} Lookup diagnostics: {details}")
        raise RuntimeError(KAGGLE_CREDENTIALS_ERROR)

def run(
    command: list[str],
    *,
    env: dict[str, str] | None = None,
    timeout_seconds: float | None = None,
) -> None:
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env, timeout=timeout_seconds)

def run_json(
    command: list[str],
    output_path: Path,
    *,
    env: dict[str, str] | None = None,
    timeout_seconds: float | None = None,
) -> dict:
    print("+", " ".join(command), ">", output_path)
    try:
        completed = subprocess.run(
            command,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout_seconds,
        )
    except subprocess.TimeoutExpired as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr, file=sys.stderr)
        timeout_label = (
            f"{timeout_seconds:g}" if timeout_seconds is not None else "unknown"
        )
        raise TimeoutError(
            f"Command exceeded {timeout_label}s: {' '.join(command)}"
        ) from exc
    if completed.returncode:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
        completed.check_returncode()
    data = json.loads(completed.stdout)
    output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n")
    print(f"Wrote {output_path}")
    return data

EMBEDDED_BOUNDED_JOB_SYNC_CODE = r'''
from __future__ import annotations

import asyncio
from datetime import UTC, datetime, timedelta
import json
from pathlib import Path
import sqlite3
import sys

from openopps.ingest import sync_jobs
from openopps.metrics import SyncMetrics
from openopps.route_registry import BoardRouteRegistry
from openopps.settings import OpenOppsSettings
from openopps.storage import OpenOppsStore


def parse_dt(value):
    if not value:
        return None
    parsed = datetime.fromisoformat(str(value))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=UTC)
    return parsed


def latest_job_syncs(db_path: Path):
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            "SELECT board_key, provider_id, max(synced_at) "
            "FROM job_sync_runs "
            "WHERE success = 1 "
            "GROUP BY board_key, provider_id"
        ).fetchall()
    return {
        (board_key, provider_id): parse_dt(synced_at)
        for board_key, provider_id, synced_at in rows
        if synced_at
    }


def route_sync_key(entry):
    return (entry.route.board_key, entry.route.provider_id)


def route_priority(item):
    index, entry, synced_at = item
    earliest = datetime.min.replace(tzinfo=UTC)
    return (
        0 if synced_at is None else 1,
        synced_at or earliest,
        entry.route.provider_id,
        entry.board.key,
        index,
    )


def selected_routes(store, db_path: Path, freshness_seconds: float, route_limit: int):
    selection = BoardRouteRegistry(store).select(ready_only=True)
    latest = latest_job_syncs(db_path)
    cutoff = datetime.now(UTC) - timedelta(seconds=freshness_seconds)
    fresh_skipped = 0
    stale = []
    for index, entry in enumerate(selection.entries):
        synced_at = latest.get(route_sync_key(entry))
        if freshness_seconds > 0 and synced_at and synced_at >= cutoff:
            fresh_skipped += 1
            continue
        stale.append((index, entry, synced_at))
    stale.sort(key=route_priority)
    selected = stale[:route_limit]
    deferred = max(0, len(stale) - len(selected))
    return [entry for _, entry, _ in selected], fresh_skipped, deferred, selection


def add_metrics(total: SyncMetrics, item: SyncMetrics) -> None:
    total.pages += item.pages
    total.boards += item.boards
    total.board_providers += item.board_providers
    total.jobs += item.jobs
    total.jobs_persisted += item.jobs_persisted
    total.job_sync_runs += item.job_sync_runs
    total.jobs_deduped += item.jobs_deduped
    total.skipped += item.skipped
    total.duplicate_routes_skipped += item.duplicate_routes_skipped
    total.retries += item.retries
    for provider_id, count in item.provider_errors.items():
        total.provider_errors[provider_id] = (
            total.provider_errors.get(provider_id, 0) + count
        )
    for provider_id, details in item.provider_error_details.items():
        total_details = total.provider_error_details.setdefault(provider_id, {})
        for reason, count in details.items():
            total_details[reason] = total_details.get(reason, 0) + count


async def main() -> None:
    db_path = Path(sys.argv[1])
    output_path = Path(sys.argv[2])
    freshness_seconds = float(sys.argv[3])
    route_limit = int(sys.argv[4])
    settings = OpenOppsSettings()
    store = OpenOppsStore(settings)
    routes, fresh_skipped, deferred, selection = selected_routes(
        store, db_path, freshness_seconds, route_limit
    )
    metrics = SyncMetrics(name="jobs.sync")
    metrics.skipped += fresh_skipped + deferred
    metrics.duplicate_routes_skipped += len(selection.duplicate_routes)
    for entry in routes:
        add_metrics(
            metrics,
            await sync_jobs(
                settings=settings,
                store=store,
                board_key=entry.route.board_key,
                provider_id=entry.route.provider_id,
            ),
        )
    data = metrics.finish().as_dict()
    data["selectedRoutes"] = len(routes)
    data["freshRoutesSkipped"] = fresh_skipped
    data["deferredRoutes"] = deferred
    data["missingRouteMetadataSkipped"] = len(selection.missing_route_metadata)
    data["compatibilityMode"] = "embedded-bounded-job-sync"
    output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n")


asyncio.run(main())
'''

def openopps_cli_supports_bounded_jobs_sync(env: dict[str, str]) -> bool:
    completed = subprocess.run(
        ["openopps", "jobs", "sync", "--help"],
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    help_text = completed.stdout + completed.stderr
    return completed.returncode == 0 and "--freshness-seconds" in help_text and "--limit" in help_text

def run_embedded_bounded_job_sync(
    output_path: Path,
    *,
    env: dict[str, str],
    timeout_seconds: float | None,
) -> dict:
    freshness_seconds = env.get("OPENOPPS_JOB_ROUTE_FRESHNESS_SECONDS", "86400")
    command = [
        sys.executable,
        "-c",
        EMBEDDED_BOUNDED_JOB_SYNC_CODE,
        str(DB_PATH),
        str(output_path),
        freshness_seconds,
        str(KAGGLE_JOB_ROUTE_LIMIT),
    ]
    print("+", sys.executable, "-c", "EMBEDDED_BOUNDED_JOB_SYNC_CODE", ">", output_path)
    completed = subprocess.run(
        command,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=timeout_seconds,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()
    data = json.loads(output_path.read_text())
    print(f"Wrote embedded bounded sync metrics to {output_path}")
    return data

def run_sync_metrics(
    output_path: Path,
    *,
    env: dict[str, str],
    timeout_seconds: float | None,
) -> dict:
    command = [
        "openopps",
        "jobs",
        "sync",
        "--metrics-json",
        "--freshness-seconds",
        env.get("OPENOPPS_JOB_ROUTE_FRESHNESS_SECONDS", "86400"),
        "--limit",
        str(KAGGLE_JOB_ROUTE_LIMIT),
    ]
    if not openopps_cli_supports_bounded_jobs_sync(env):
        print(
            "Installed OpenOpps CLI does not expose bounded jobs sync flags; "
            "using embedded bounded job sync."
        )
        return run_embedded_bounded_job_sync(
            output_path,
            env=env,
            timeout_seconds=timeout_seconds,
        )
    try:
        return run_json(
            command,
            output_path,
            env=env,
            timeout_seconds=timeout_seconds,
        )
    except subprocess.CalledProcessError:
        print(
            "bounded openopps jobs sync --metrics-json failed; falling back to "
            "plain bounded jobs sync and SQLite-derived metrics."
        )
        run(
            [part for part in command if part != "--metrics-json"],
            env=env,
            timeout_seconds=timeout_seconds,
        )
        data = sqlite_sync_metrics(DB_PATH)
        output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n")
        print(f"Wrote compatibility sync metrics to {output_path}")
        return data

def sqlite_sync_metrics(db_path: Path) -> dict:
    def count(conn, table: str, where: str | None = None) -> int:
        try:
            query = f"SELECT count(*) FROM {table}"
            if where:
                query = f"{query} WHERE {where}"
            return int(conn.execute(query).fetchone()[0])
        except sqlite3.Error:
            return 0

    with sqlite3.connect(db_path) as conn:
        jobs = count(conn, "jobs")
        successful_runs = count(conn, "job_sync_runs", "success = 1")
        return {
            "compatibilityMode": "sqlite-derived-after-plain-sync",
            "sourcesProcessed": count(conn, "sources"),
            "boardsPersisted": count(conn, "boards"),
            "boardProviders": count(conn, "board_providers"),
            "jobsPersisted": jobs,
            "jobSyncRuns": successful_runs,
            "providerErrors": {},
            "providerErrorDetails": {},
        }

def install_openopps() -> None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", PACKAGE_SPEC, "kaggle"])

def copy_latest_input_db() -> None:
    db_candidates = sorted(KAGGLE_INPUT_DIR.glob(INPUT_DB_GLOB))
    if db_candidates:
        source_db = max(db_candidates, key=lambda path: path.stat().st_mtime)
        shutil.copy2(source_db, DB_PATH)
        print(f"Copied prior OpenOpps DB snapshot from {source_db} to {DB_PATH}")
        restore_projected_sqlite_columns_from_input_exports()
    else:
        print("No prior OpenOpps DB snapshot found; creating a new ledger.")

def quote_identifier(value: str) -> str:
    return '"' + value.replace('"', '""') + '"'

def restore_projected_sqlite_table_columns(
    *,
    parquet_glob: str,
    table_name: str,
    column_names: list[str],
) -> None:
    parquet_candidates = sorted(KAGGLE_INPUT_DIR.glob(parquet_glob))
    if not parquet_candidates or not DB_PATH.exists():
        return
    source_parquet = max(parquet_candidates, key=lambda path: path.stat().st_mtime)
    table_sql = quote_identifier(table_name)
    missing_condition = " OR ".join(
        f"{quote_identifier(column)} IS NULL" for column in column_names
    )
    with sqlite3.connect(DB_PATH) as conn:
        missing_count = int(
            conn.execute(
                f"SELECT count(*) FROM {table_sql} WHERE {missing_condition}"
            ).fetchone()[0]
        )
    if missing_count == 0:
        return

    import polars as pl

    restore_csv = OUTPUT_DIR / f"_restore_{table_name}.csv"
    restore_columns = ["id", *column_names]
    pl.scan_parquet(source_parquet).select(restore_columns).sink_csv(restore_csv)
    restored_rows = 0
    csv.field_size_limit(sys.maxsize)
    with sqlite3.connect(DB_PATH) as conn, restore_csv.open(
        newline="", encoding="utf-8"
    ) as handle:
        reader = csv.DictReader(handle)
        restore_table = f"restore_{table_name}"
        restore_table_sql = quote_identifier(restore_table)
        column_defs = ", ".join(
            f"{quote_identifier(column)} TEXT" for column in restore_columns
        )
        conn.execute(f"CREATE TEMP TABLE {restore_table_sql} ({column_defs}, PRIMARY KEY (id))")
        batch = []
        for row in reader:
            batch.append(tuple(row[column] for column in restore_columns))
            if len(batch) >= 1000:
                conn.executemany(
                    f"INSERT OR REPLACE INTO {restore_table_sql} VALUES "
                    f"({', '.join('?' for _ in restore_columns)})",
                    batch,
                )
                restored_rows += len(batch)
                batch.clear()
        if batch:
            conn.executemany(
                f"INSERT OR REPLACE INTO {restore_table_sql} VALUES "
                f"({', '.join('?' for _ in restore_columns)})",
                batch,
            )
            restored_rows += len(batch)
        assignments = ", ".join(
            f"{quote_identifier(column)} = ("
            f"SELECT {restore_table_sql}.{quote_identifier(column)} "
            f"FROM {restore_table_sql} "
            f"WHERE {restore_table_sql}.id = {table_sql}.id)"
            for column in column_names
        )
        conn.execute(
            f"UPDATE {table_sql} SET {assignments} "
            f"WHERE {missing_condition} AND EXISTS ("
            f"SELECT 1 FROM {restore_table_sql} WHERE {restore_table_sql}.id = {table_sql}.id)"
        )
        conn.commit()
    restore_csv.unlink(missing_ok=True)
    print(
        "Restored projected SQLite values from prior Parquet export:",
        json.dumps(
            {
                "source": str(source_parquet),
                "table": table_name,
                "columns": column_names,
                "missingBefore": missing_count,
                "restoreRows": restored_rows,
            },
            sort_keys=True,
        ),
    )

def restore_projected_sqlite_columns_from_input_exports() -> None:
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_JOB_VERSIONS_PARQUET_GLOB,
        table_name="job_versions",
        column_names=["description_html", "job_description"],
    )
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_JOB_PAYLOAD_SNAPSHOTS_PARQUET_GLOB,
        table_name="job_payload_snapshots",
        column_names=["payload"],
    )

def download_dataset_assets() -> None:
    urllib.request.urlretrieve(DATASET_IMAGE_URL, OUTPUT_DIR / "dataset-cover-image.png")

def slugify(value: str) -> str:
    slug = SLUG_RE.sub("-", value.lower()).strip("-")
    return slug or hashlib.sha1(value.encode("utf-8")).hexdigest()[:12]

def stable_id(*parts) -> str:
    visible = ":".join(
        slugify(str(part)) for part in parts if part is not None and str(part) != ""
    )
    if len(visible) <= 180:
        return visible
    digest = hashlib.sha1(visible.encode("utf-8")).hexdigest()[:16]
    return f"{visible[:120]}-{digest}"

def stable_id_from_slugs(*slugs: str) -> str:
    visible = ":".join(slug for slug in slugs if slug)
    if len(visible) <= 180:
        return visible
    digest = hashlib.sha1(visible.encode("utf-8")).hexdigest()[:16]
    return f"{visible[:120]}-{digest}"

@lru_cache(maxsize=512)
def cached_slug(value: str) -> str:
    return slugify(value)

def strip_html(value: str | None) -> str | None:
    if not value:
        return None
    with_breaks = re.sub(r"(?i)<\s*br\s*/?\s*>", "\n", value)
    with_breaks = re.sub(r"(?i)</\s*(p|div|li|h[1-6])\s*>", "\n", with_breaks)
    text = re.sub(r"<[^>]+>", " ", with_breaks)
    text = unescape(text)
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n\s+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()
    return text or None

def normalized_skill_text(values) -> str:
    raw = " ".join(str(value)[:SKILL_TEXT_VALUE_LIMIT] for value in values if value)
    raw = strip_html(raw) or raw
    normalized = re.sub(r"[^a-z0-9+#]+", " ", raw.casefold())
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return f" {normalized} "

@lru_cache(maxsize=256)
def compile_skill_aliases(aliases: tuple[str, ...]):
    normalized_aliases = tuple(
        sorted(
            {
                normalized_alias
                for alias in aliases
                if (normalized_alias := normalized_skill_text([alias]).strip())
            },
            key=len,
            reverse=True,
        )
    )
    if not normalized_aliases:
        return frozenset(), ()
    single_tokens = frozenset(
        normalized_alias
        for normalized_alias in normalized_aliases
        if " " not in normalized_alias
    )
    phrases = tuple(
        normalized_alias
        for normalized_alias in normalized_aliases
        if " " in normalized_alias
    )
    return single_tokens, phrases

@lru_cache(maxsize=1)
def compiled_skill_catalog():
    return tuple(
        (
            group_name,
            tuple(
                (
                    keyword,
                    *compile_skill_aliases(tuple(aliases)),
                )
                for keyword, aliases in keywords
            ),
        )
        for group_name, keywords in SKILL_CATALOG
    )

@lru_cache(maxsize=1)
def compiled_level_aliases():
    return tuple(
        (label, *compile_skill_aliases(aliases))
        for label, aliases in SKILL_LEVEL_ALIASES
    )

def has_compiled_skill_alias(
    normalized_text: str,
    text_tokens: frozenset[str],
    single_tokens: frozenset[str],
    phrases: tuple[str, ...],
) -> bool:
    return (not text_tokens.isdisjoint(single_tokens)) or any(
        phrase in normalized_text for phrase in phrases
    )

def json_value(value):
    if value in (None, ""):
        return None
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return None
    return value

def json_list(value) -> list:
    data = json_value(value)
    return data if isinstance(data, list) else []

def string_or_none(value) -> str | None:
    if value is None:
        return None
    return str(value)

def skill_level(row) -> str | None:
    text = normalized_skill_text([row["experience"], row["title"]])
    text_tokens = frozenset(text.split())
    for label, single_tokens, phrases in compiled_level_aliases():
        if has_compiled_skill_alias(text, text_tokens, single_tokens, phrases):
            return label
    return row["experience"]

def extract_version_skills(row) -> list[dict]:
    existing = json_list(row["skills"])
    if existing:
        return existing
    description_text = row["description"] or row["description_html"]
    text = normalized_skill_text(
        [
            row["title"],
            row["department"],
            row["team"],
            row["employment_type"],
            description_text,
            *json_list(row["responsibilities"]),
            *json_list(row["qualifications"]),
        ]
    )
    if not text.strip():
        return []
    level = skill_level(row)
    text_tokens = frozenset(text.split())
    skills = []
    for group_name, keywords in compiled_skill_catalog():
        matched = [
            keyword
            for keyword, single_tokens, phrases in keywords
            if has_compiled_skill_alias(text, text_tokens, single_tokens, phrases)
        ]
        if matched:
            skills.append({"name": group_name, "level": level, "keywords": matched[:12]})
    return skills

def backfill_openopps_skill_tables(db_path: Path) -> dict[str, int]:
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA busy_timeout = 30000")
        conn.execute("PRAGMA temp_store = MEMORY")
        tables = {
            row[0]
            for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")
        }
        required = {
            "jobs",
            "job_versions",
            "job_version_skills",
            "job_version_skill_keywords",
        }
        if not required <= tables:
            return {
                "versionsExamined": 0,
                "versionsBackfilled": 0,
                "skillsInserted": 0,
                "skillKeywordsInserted": 0,
            }

        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS ix_job_version_skills_job_version_id
            ON job_version_skills (job_version_id)
            """
        )
        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS ix_job_version_skill_keywords_skill_id
            ON job_version_skill_keywords (skill_id)
            """
        )
        examined = 0
        backfilled = 0
        skills_inserted = 0
        keywords_inserted = 0
        last_rowid = 0
        chunk_size = 2000
        while True:
            rows = conn.execute(
                """
                SELECT
                    v.rowid AS _rowid,
                    v.id,
                    v.title,
                    v.department,
                    v.team,
                    v.employment_type,
                    v.description,
                    v.description_html,
                    v.experience,
                    v.responsibilities,
                    v.qualifications,
                    v.skills
                FROM job_versions AS v
                WHERE v.rowid > ?
                  AND (
                    NOT EXISTS (
                        SELECT 1
                        FROM job_version_skills AS s
                        WHERE s.job_version_id = v.id
                    )
                    OR EXISTS (
                        SELECT 1
                        FROM job_version_skills AS s
                        WHERE s.job_version_id = v.id
                          AND NOT EXISTS (
                            SELECT 1
                            FROM job_version_skill_keywords AS k
                            WHERE k.skill_id = s.id
                          )
                    )
                  )
                ORDER BY v.rowid
                LIMIT ?
                """,
                (last_rowid, chunk_size),
            ).fetchall()
            if not rows:
                break
            last_rowid = int(rows[-1]["_rowid"])
            version_ids = [(row["id"],) for row in rows]
            conn.executemany(
                """
                DELETE FROM job_version_skill_keywords
                WHERE skill_id IN (
                    SELECT id
                    FROM job_version_skills
                    WHERE job_version_id = ?
                )
                """,
                version_ids,
            )
            conn.executemany(
                "DELETE FROM job_version_skills WHERE job_version_id = ?",
                version_ids,
            )
            version_updates = []
            skill_rows = []
            keyword_rows = []
            for row in rows:
                examined += 1
                skills = extract_version_skills(row)
                if not skills:
                    continue
                version_slug = slugify(str(row["id"]))
                if not json_list(row["skills"]):
                    version_updates.append(
                        (
                            json.dumps(
                                skills,
                                sort_keys=True,
                                separators=(",", ":"),
                            ),
                            row["id"],
                        )
                    )
                for ordinal, skill in enumerate(skills):
                    skill_id = stable_id_from_slugs(
                        version_slug,
                        "skill",
                        str(ordinal),
                    )
                    skill_slug = slugify(skill_id)
                    skill_rows.append(
                        (
                            skill_id,
                            row["id"],
                            ordinal,
                            string_or_none(skill.get("name")),
                            string_or_none(skill.get("level")),
                        )
                    )
                    for keyword_ordinal, keyword in enumerate(skill.get("keywords") or []):
                        keyword_text = str(keyword)
                        keyword_rows.append(
                            (
                                stable_id_from_slugs(
                                    skill_slug,
                                    "keyword",
                                    str(keyword_ordinal),
                                    cached_slug(keyword_text),
                                ),
                                skill_id,
                                keyword_ordinal,
                                keyword_text,
                            )
                        )
                backfilled += 1
            if version_updates:
                conn.executemany(
                    "UPDATE job_versions SET skills = ? WHERE id = ?",
                    version_updates,
                )
            if skill_rows:
                cursor = conn.executemany(
                    """
                    INSERT INTO job_version_skills (
                        id,
                        job_version_id,
                        ordinal,
                        name,
                        level
                    ) VALUES (?, ?, ?, ?, ?)
                    """,
                    skill_rows,
                )
                if cursor.rowcount and cursor.rowcount > 0:
                    skills_inserted += cursor.rowcount
            if keyword_rows:
                cursor = conn.executemany(
                    """
                    INSERT INTO job_version_skill_keywords (
                        id,
                        skill_id,
                        ordinal,
                        keyword
                    ) VALUES (?, ?, ?, ?)
                    """,
                    keyword_rows,
                )
                if cursor.rowcount and cursor.rowcount > 0:
                    keywords_inserted += cursor.rowcount
            conn.commit()
        conn.commit()
    return {
        "versionsExamined": examined,
        "versionsBackfilled": backfilled,
        "skillsInserted": skills_inserted,
        "skillKeywordsInserted": keywords_inserted,
    }

def write_json(path: Path, data: dict | list) -> None:
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n")
    print(f"Wrote {path}")

def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text())

def checkpoint_sqlite(db_path: Path) -> None:
    if db_path.exists():
        with sqlite3.connect(db_path) as conn:
            conn.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchall()
    for suffix in ("-journal", "-shm", "-wal"):
        sidecar = db_path.with_name(f"{db_path.name}{suffix}")
        if sidecar.exists():
            sidecar.unlink()

def sqlite_header_read_write_versions(db_path: Path) -> tuple[int, int]:
    header = db_path.read_bytes()[:20]
    if len(header) < 20 or not header.startswith(b"SQLite format 3\x00"):
        raise RuntimeError(f"Not a SQLite database file: {db_path}")
    return header[18], header[19]

def assert_portable_sqlite_upload(db_path: Path) -> None:
    versions = sqlite_header_read_write_versions(db_path)
    if versions != (1, 1):
        raise RuntimeError(
            "SQLite upload copy is not in portable rollback-journal format: "
            f"header read/write versions are {versions}"
        )
    with sqlite3.connect(f"file:{db_path}?mode=ro&immutable=1", uri=True) as conn:
        quick_check = str(conn.execute("PRAGMA quick_check").fetchone()[0])
        if quick_check.lower() != "ok":
            raise RuntimeError(f"SQLite upload copy failed quick_check: {quick_check}")
        table_count = int(
            conn.execute(
                "SELECT count(*) FROM sqlite_master WHERE type = 'table'"
            ).fetchone()[0]
        )
    if table_count == 0:
        raise RuntimeError("SQLite upload copy has no readable tables.")

def finalize_sqlite_for_upload(db_path: Path) -> None:
    if not db_path.exists():
        return
    portable_db = db_path.with_name(f".{db_path.name}.portable")
    for suffix in ("-journal", "-shm", "-wal"):
        sidecar = portable_db.with_name(f"{portable_db.name}{suffix}")
        if sidecar.exists():
            sidecar.unlink()
    if portable_db.exists():
        portable_db.unlink()
    with sqlite3.connect(db_path) as conn:
        checkpoint = conn.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone()
        if checkpoint and int(checkpoint[0]) != 0:
            raise RuntimeError(f"SQLite upload copy has busy WAL readers: {checkpoint}")
        literal = "'" + portable_db.as_posix().replace("'", "''") + "'"
        conn.execute(f"VACUUM INTO {literal}")
    with sqlite3.connect(portable_db) as conn:
        journal_mode = str(conn.execute("PRAGMA journal_mode=DELETE").fetchone()[0])
    assert_portable_sqlite_upload(portable_db)
    portable_db.replace(db_path)
    checkpoint_sqlite(db_path)
    if journal_mode.lower() != "delete":
        raise RuntimeError(
            f"SQLite upload copy did not switch to DELETE journal mode: {journal_mode}"
        )
    assert_portable_sqlite_upload(db_path)

def write_dataset_metadata() -> None:
    write_json(OUTPUT_DIR / "dataset-metadata.json", DATASET_METADATA)

def write_sqlite_metadata(db_path: Path) -> None:
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS openopps_tables (
                table_name TEXT PRIMARY KEY,
                table_title TEXT NOT NULL,
                table_description TEXT NOT NULL,
                csv_path TEXT NOT NULL,
                parquet_path TEXT NOT NULL
            )
            """
        )
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS openopps_columns (
                table_name TEXT NOT NULL,
                column_name TEXT NOT NULL,
                column_title TEXT NOT NULL,
                column_description TEXT NOT NULL,
                logical_type TEXT NOT NULL,
                json_schema_type TEXT NOT NULL,
                required INTEGER NOT NULL,
                source_name TEXT,
                format TEXT,
                enum_json TEXT,
                examples_json TEXT,
                default_json TEXT,
                PRIMARY KEY (table_name, column_name)
            )
            """
        )
        conn.execute("DELETE FROM openopps_columns")
        conn.execute("DELETE FROM openopps_tables")
        conn.executemany(
            """
            INSERT INTO openopps_tables (
                table_name,
                table_title,
                table_description,
                csv_path,
                parquet_path
            ) VALUES (
                :table_name,
                :table_title,
                :table_description,
                :csv_path,
                :parquet_path
            )
            """,
            OPENOPPS_TABLE_ROWS,
        )
        conn.executemany(
            """
            INSERT INTO openopps_columns (
                table_name,
                column_name,
                column_title,
                column_description,
                logical_type,
                json_schema_type,
                required,
                source_name,
                format,
                enum_json,
                examples_json,
                default_json
            ) VALUES (
                :table_name,
                :column_name,
                :column_title,
                :column_description,
                :logical_type,
                :json_schema_type,
                :required,
                :source_name,
                :format,
                :enum_json,
                :examples_json,
                :default_json
            )
            """,
            OPENOPPS_COLUMN_ROWS,
        )

def write_table_csv(conn: sqlite3.Connection, table_name: str, csv_path: Path) -> None:
    cursor = conn.execute(f'SELECT * FROM "{table_name}"')
    headers = [column[0] for column in cursor.description]
    with csv_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(headers)
        while rows := cursor.fetchmany(10_000):
            writer.writerows(rows)

def write_full_table_exports(db_path: Path) -> None:
    import polars as pl

    csv_dir = OUTPUT_DIR / CSV_DIR
    parquet_dir = OUTPUT_DIR / PARQUET_DIR
    csv_dir.mkdir(parents=True, exist_ok=True)
    parquet_dir.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(db_path) as conn:
        for table in OPENOPPS_TABLE_ROWS:
            table_name = table["table_name"]
            csv_path = csv_dir / f"{table_name}.csv"
            parquet_path = parquet_dir / f"{table_name}.parquet"
            print(f"Exporting {table_name}...", flush=True)
            write_table_csv(conn, table_name, csv_path)
            pl.scan_csv(
                csv_path,
                infer_schema_length=1000,
                low_memory=True,
            ).sink_parquet(parquet_path)

def project_sqlite_for_kaggle_indexer(db_path: Path) -> dict:
    projection_columns = (
        ("job_versions", "description_html"),
        ("job_versions", "job_description"),
        ("job_payload_snapshots", "payload"),
    )
    total_rows = 0
    total_bytes = 0
    nulled_columns = []
    with sqlite3.connect(db_path) as conn:
        for table_name, column_name in projection_columns:
            row = conn.execute(
                f"""
                SELECT COUNT(*), COALESCE(SUM(length(CAST("{column_name}" AS blob))), 0)
                FROM "{table_name}"
                WHERE "{column_name}" IS NOT NULL
                """
            ).fetchone()
            rows = int(row[0] or 0)
            bytes_removed = int(row[1] or 0)
            if rows:
                conn.execute(
                    f'UPDATE "{table_name}" SET "{column_name}" = NULL '
                    f'WHERE "{column_name}" IS NOT NULL'
                )
                nulled_columns.append(f"{table_name}.{column_name}")
                total_rows += rows
                total_bytes += bytes_removed
        conn.commit()
    if nulled_columns:
        print(
            "Prepared SQLite upload projection for Kaggle indexer:",
            json.dumps(
                {
                    "nulledColumns": nulled_columns,
                    "rows": total_rows,
                    "estimatedBytesRemoved": total_bytes,
                },
                sort_keys=True,
            ),
            flush=True,
        )
    return {"projected_rows": total_rows, "estimated_bytes_removed": total_bytes}

def normalize_sqlite_schema_for_kaggle_indexer(db_path: Path) -> int:
    replacements = (
        (re.compile(r"\bVARCHAR(?:\(\d+\))?\b"), "TEXT"),
        (re.compile(r"\bJSON\b"), "TEXT"),
        (re.compile(r"\bDATETIME\b"), "TEXT"),
        (re.compile(r"\bBOOLEAN\b"), "INTEGER"),
        (re.compile(r"\bFLOAT\b"), "REAL"),
    )
    updated = 0
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            "SELECT rowid, sql FROM sqlite_schema WHERE type = 'table' AND sql IS NOT NULL"
        ).fetchall()
        conn.execute("PRAGMA writable_schema = ON")
        try:
            for rowid, sql in rows:
                normalized = str(sql)
                for pattern, replacement in replacements:
                    normalized = pattern.sub(replacement, normalized)
                if normalized != sql:
                    conn.execute(
                        "UPDATE sqlite_schema SET sql = ? WHERE rowid = ?",
                        (normalized, rowid),
                    )
                    updated += 1
            if updated:
                schema_version = int(conn.execute("PRAGMA schema_version").fetchone()[0])
                conn.execute(f"PRAGMA schema_version = {schema_version + 1}")
        finally:
            conn.execute("PRAGMA writable_schema = OFF")
        conn.commit()
        integrity = str(conn.execute("PRAGMA integrity_check").fetchone()[0])
    if integrity.lower() != "ok":
        raise RuntimeError(
            f"SQLite schema normalization failed integrity_check: {integrity}"
        )
    if updated:
        print(
            "Normalized SQLite upload schema for Kaggle indexer:",
            json.dumps({"tables": updated}, sort_keys=True),
            flush=True,
        )
    return updated

def table_count(conn: sqlite3.Connection, table_name: str) -> int:
    try:
        return int(conn.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0])
    except sqlite3.Error:
        return 0

def snapshot_quality_report() -> dict:
    hard_blockers = []
    counts = {}
    with sqlite3.connect(DB_PATH) as conn:
        for table in OPENOPPS_TABLE_ROWS:
            table_name = table["table_name"]
            counts[table_name] = table_count(conn, table_name)

    required_paths = [
        "dataset-cover-image.png",
        "dataset-metadata.json",
        *PUBLIC_UPLOAD_DATA_FILES,
    ]
    required_files = []
    for relative_path in required_paths:
        path = OUTPUT_DIR / relative_path
        item = {
            "path": relative_path,
            "exists": path.exists(),
            "sizeBytes": path.stat().st_size if path.exists() else 0,
        }
        required_files.append(item)
        if not item["exists"]:
            hard_blockers.append(f"missing_required_file:{relative_path}")
        elif item["sizeBytes"] == 0:
            hard_blockers.append(f"empty_required_file:{relative_path}")

    if counts.get("jobs", 0) == 0 and not os.environ.get("OPENOPPS_EMPTY_SNAPSHOT_EXPLANATION"):
        hard_blockers.append("missing_current_job_evidence")
    if counts.get("job_versions", 0) > 0:
        if counts.get("job_version_skills", 0) == 0:
            hard_blockers.append("missing_job_version_skill_rows")
        if counts.get("job_version_skill_keywords", 0) == 0:
            hard_blockers.append("missing_job_version_skill_keyword_rows")
    if counts.get("openopps_tables") != len(OPENOPPS_TABLE_ROWS):
        hard_blockers.append("missing_openopps_table_metadata")
    if counts.get("openopps_columns", 0) <= 0:
        hard_blockers.append("missing_openopps_column_metadata")

    return {
        "generatedAt": datetime.now(UTC).isoformat(),
        "status": "fail" if hard_blockers else "pass",
        "hardBlockers": hard_blockers,
        "warnings": [],
        "counts": counts,
        "requiredFiles": required_files,
        "syncMetrics": read_json(OUTPUT_DIR / "sync_metrics.json"),
        "statusSummary": read_json(OUTPUT_DIR / "status.json"),
        "coverageSummary": read_json(OUTPUT_DIR / "coverage.json"),
    }

def prune_private_upload_files() -> None:
    for relative_path in (
        "sync_metrics.json",
        "status.json",
        "coverage.json",
        "snapshot-quality.json",
        "sync_stderr.txt",
        "generate_kaggle_metadata.py",
    ):
        path = OUTPUT_DIR / relative_path
        if path.exists():
            path.unlink()
    shutil.rmtree(OUTPUT_DIR / "_manager-unused", ignore_errors=True)
    project_sqlite_for_kaggle_indexer(DB_PATH)
    normalize_sqlite_schema_for_kaggle_indexer(DB_PATH)
    finalize_sqlite_for_upload(DB_PATH)

def write_public_bundle() -> dict:
    write_dataset_metadata()
    write_sqlite_metadata(DB_PATH)
    checkpoint_sqlite(DB_PATH)
    write_full_table_exports(DB_PATH)
    project_sqlite_for_kaggle_indexer(DB_PATH)
    normalize_sqlite_schema_for_kaggle_indexer(DB_PATH)
    finalize_sqlite_for_upload(DB_PATH)
    quality = snapshot_quality_report()
    write_json(OUTPUT_DIR / "snapshot-quality.json", quality)
    if quality["status"] != "pass":
        blockers = "; ".join(quality["hardBlockers"]) or "unknown quality failure"
        raise RuntimeError(f"Snapshot quality gate failed: {blockers}")
    prune_private_upload_files()
    return quality

def update_kaggle_dataset_file_metadata(dataset_basics: dict | None = None) -> None:
    from kaggle.api.kaggle_api_extended import KaggleApi
    from kagglesdk.datasets.types.dataset_api_service import (
        ApiUpdateDatasetMetadataRequest,
    )
    from kagglesdk.datasets.types.dataset_types import (
        DatasetSettings,
        DatasetSettingsFile,
        DatasetSettingsFileColumn,
    )

    metadata_path = OUTPUT_DIR / "dataset-metadata.json"
    metadata = json.loads(metadata_path.read_text())
    resources = metadata.get("resources") or []
    if not resources:
        raise RuntimeError(f"No Kaggle resources found in {metadata_path}")

    api = KaggleApi()
    api.authenticate()

    settings = DatasetSettings()
    settings.title = str(metadata.get("title") or "")
    settings.subtitle = str(metadata.get("subtitle") or "")
    settings.description = str(metadata.get("description") or "")
    settings.is_private = bool(metadata.get("isPrivate", False))
    settings.licenses = [
        api._new_license(str(license_data["name"]))
        for license_data in metadata.get("licenses", [])
        if license_data.get("name")
    ]
    settings.keywords = [str(keyword) for keyword in metadata.get("keywords", [])]
    settings.expected_update_frequency = str(
        metadata.get("expectedUpdateFrequency") or "not specified"
    )
    settings.user_specified_sources = str(metadata.get("userSpecifiedSources") or "")
    settings.data = [
        _dataset_settings_file(
            resource,
            DatasetSettingsFile,
            DatasetSettingsFileColumn,
            base_dir=OUTPUT_DIR,
        )
        for resource in resources
    ]

    owner_slug, dataset_slug = str(metadata.get("id") or DATASET_ID).split("/", 1)
    request = ApiUpdateDatasetMetadataRequest()
    request.owner_slug = owner_slug
    request.dataset_slug = dataset_slug
    request.settings = settings

    try:
        with api.build_kaggle_client() as kaggle:
            response = kaggle.datasets.dataset_api_client.update_dataset_metadata(request)
        errors = getattr(response, "errors", None) or []
        if errors:
            raise RuntimeError(f"Kaggle dataset metadata update failed: {errors}")
        print(f"Updated Kaggle public dataset metadata for {len(settings.data or [])} public files.")
    except Exception as exc:
        print(
            "Kaggle public dataset metadata update failed; continuing with "
            f"live databundle metadata repair: {type(exc).__name__}"
        )

    if dataset_basics is None:
        session, headers = kaggle_internal_metadata_session()
        dataset_basics = kaggle_dataset_basics(session, headers)
    update_kaggle_databundle_metadata_external(metadata, dataset_basics)

def _dataset_settings_file(
    resource,
    dataset_settings_file_cls,
    dataset_settings_file_column_cls,
    *,
    base_dir=None,
):
    file_metadata = dataset_settings_file_cls()
    file_metadata.name = str(resource["path"])
    file_metadata.description = str(resource.get("description") or "")
    if base_dir is not None:
        file_path = Path(base_dir) / str(resource["path"])
        if file_path.exists():
            file_metadata.total_bytes = file_path.stat().st_size
    columns = []
    for field in resource.get("schema", {}).get("fields", []):
        column = dataset_settings_file_column_cls()
        column.name = str(field["name"])
        column.description = str(field.get("description") or "")
        column.type = str(field.get("type") or "")
        columns.append(column)
    file_metadata.columns = columns
    return file_metadata

def kaggle_basic_auth_header() -> str:
    username = os.environ.get("KAGGLE_USERNAME", "").strip()
    key = os.environ.get("KAGGLE_KEY", "").strip()
    token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
    if (not username or not key) and token:
        try:
            token_data = json.loads(token)
        except json.JSONDecodeError:
            token_data = {}
        username = username or str(token_data.get("username") or "").strip()
        key = key or str(token_data.get("key") or "").strip()
    if not username or not key:
        raise RuntimeError("Kaggle username/key credentials are required for metadata repair.")
    encoded = base64.b64encode(f"{username}:{key}".encode()).decode()
    return f"Basic {encoded}"

def kaggle_internal_metadata_session():
    import requests

    owner_slug, dataset_slug = DATASET_ID.split("/", 1)
    session = requests.Session()
    response = session.get(
        f"https://www.kaggle.com/datasets/{owner_slug}/{dataset_slug}",
        timeout=60,
    )
    response.raise_for_status()
    xsrf_token = session.cookies.get("XSRF-TOKEN")
    if not xsrf_token:
        raise RuntimeError("Kaggle XSRF token cookie was not returned for metadata repair.")
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Authorization": kaggle_basic_auth_header(),
        "X-XSRF-TOKEN": xsrf_token,
    }
    return session, headers

def kaggle_internal_post(session, headers: dict[str, str], route: str, body: dict) -> dict:
    response = session.post(
        f"https://www.kaggle.com/api/i/{route}",
        headers=headers,
        json=body,
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(
            f"Kaggle internal metadata API failed for {route}: "
            f"{response.status_code} {response.text[:500]}"
        )
    return response.json()

def kaggle_dataset_basics(
    session,
    headers: dict[str, str],
    *,
    dataset_version_number: int | None = None,
) -> dict:
    owner_slug, dataset_slug = DATASET_ID.split("/", 1)
    body = {
        "ownerSlug": owner_slug,
        "datasetSlug": dataset_slug,
    }
    if dataset_version_number is not None:
        body["datasetVersionNumber"] = dataset_version_number
    return kaggle_internal_post(
        session,
        headers,
        "datasets.DatasetDetailService/GetDatasetBasics",
        body,
    )

def wait_for_new_live_dataset_version(previous_version: int | None) -> dict:
    session, headers = kaggle_internal_metadata_session()
    deadline = time.time() + float(os.environ.get("OPENOPPS_KAGGLE_METADATA_WAIT_SECONDS", "900"))
    while True:
        basics = kaggle_dataset_basics(session, headers)
        current_version = int(basics.get("datasetVersionNumber") or 0)
        data = basics.get("data") or {}
        if (
            data.get("firestorePath")
            and data.get("versionId")
            and (previous_version is None or current_version > previous_version)
        ):
            print(
                "Kaggle live dataset version ready for metadata repair:",
                json.dumps(
                    {
                        "datasetVersionNumber": current_version,
                        "datasetVersionId": basics.get("datasetVersionId"),
                        "databundleVersionId": data.get("versionId"),
                    },
                    sort_keys=True,
                ),
            )
            return basics
        if time.time() >= deadline:
            raise TimeoutError(
                "Timed out waiting for Kaggle to expose the newly published dataset version."
            )
        print(
            "Waiting for new Kaggle dataset version before metadata repair:",
            json.dumps(
                {
                    "previousVersionNumber": previous_version,
                    "currentVersionNumber": current_version,
                },
                sort_keys=True,
            ),
        )
        time.sleep(15)

def kaggle_databundle_column_type(field_type: str) -> tuple[str, str]:
    normalized = (field_type or "string").lower()
    if normalized in {"datetime", "date", "time"}:
        return "DATE_TIME", "EXTENDED_DATA_TYPE_UNSPECIFIED"
    if normalized in {"integer", "int"}:
        return "NUMERIC", "INTEGER"
    if normalized in {"numeric", "number", "float", "decimal"}:
        return "NUMERIC", "DECIMAL"
    if normalized == "boolean":
        return "BOOLEAN", "EXTENDED_DATA_TYPE_UNSPECIFIED"
    if normalized == "url":
        return "STRING", "URL"
    if normalized == "uuid":
        return "STRING", "UUID"
    if normalized == "id":
        return "STRING", "ID"
    return "STRING", "EXTENDED_DATA_TYPE_UNSPECIFIED"

def kaggle_column_type_from_field(field: dict) -> str:
    field_type = str(field.get("type") or "").lower()
    field_name = str(field.get("name") or "")
    field_format = str(field.get("format") or "")
    if field_format in {"date-time", "date"}:
        return "datetime"
    if field_format in {"uri", "url"} or field_name.endswith("_url"):
        return "url"
    if field_name == "id" or field_name.endswith("_id") or field_name.endswith("_key"):
        return "id"
    if field_type in {"boolean", "datetime", "id", "integer", "numeric", "number", "url", "uuid"}:
        return field_type
    if field_type == "string":
        return "string"
    schema_types = {
        item.strip()
        for item in str(field.get("jsonSchemaType") or "").split("|")
        if item.strip() and item.strip() != "null"
    }
    if "boolean" in schema_types:
        return "boolean"
    if "integer" in schema_types:
        return "integer"
    if "number" in schema_types:
        return "numeric"
    return "string"

def update_databundle_entity_metadata(
    post,
    verification_info: dict,
    *,
    firestore_path: str,
    description: str,
    fields: list[dict],
) -> tuple[dict, int]:
    columns = []
    fields_by_name = {str(field["name"]): field for field in fields}
    if fields_by_name:
        live_columns = post(
            "datasets.databundles.DatabundleService/GetDatabundleExternalColumns",
            {
                "verificationInfo": verification_info,
                "firestorePath": firestore_path,
            },
        ).get("columns") or []
        for live_column in live_columns:
            field = fields_by_name.get(str(live_column.get("name") or ""))
            column_type, extended_type = kaggle_databundle_column_type(
                kaggle_column_type_from_field(field or {})
            )
            column = dict(live_column)
            column.update(
                {
                    "description": str((field or {}).get("description") or ""),
                    "type": column_type,
                    "extendedType": extended_type,
                }
            )
            columns.append(column)
    response = post(
        "datasets.databundles.DatabundleService/UpdateDatabundleMetadataExternal",
        {
            "verificationInfo": verification_info,
            "firestorePath": firestore_path,
            "description": description,
            "columns": columns,
        },
    )
    return response, len(columns)

def update_sqlite_table_metadata_external(
    post,
    verification_info: dict,
    sqlite_file_info: dict,
) -> tuple[int, int, dict]:
    sqlite_info = sqlite_file_info.get("sqliteInfo") or {}
    table_count = int((sqlite_info.get("tables") or {}).get("totalChildren") or 0)
    if table_count == 0:
        raise RuntimeError(
            "Kaggle SQLite indexer did not index openoppsdb.sqlite; "
            "no sqliteInfo.tables were exposed for live table metadata repair."
        )
    children = post(
        "datasets.databundles.DatabundleService/GetDatabundleExternalChildren",
        {
            "verificationInfo": verification_info,
            "firestorePath": sqlite_file_info["path"],
            "offset": 0,
            "count": max(table_count, len(SQLITE_TABLE_METADATA), 200),
            "depth": 1,
            "enforceMaxDepthConstraint": False,
        },
    )
    live_tables = {
        str(table_info.get("name") or ""): table_info
        for table_info in children.get("tables") or []
    }
    expected_tables = {str(table["name"]): table for table in SQLITE_TABLE_METADATA}
    missing_tables = sorted(set(expected_tables) - set(live_tables))
    if missing_tables:
        raise RuntimeError(
            "Kaggle SQLite indexer omitted expected openoppsdb tables: "
            + ", ".join(missing_tables)
        )
    updated_tables = 0
    updated_columns = 0
    rating = {}
    for table_name, table_metadata in expected_tables.items():
        live_table = live_tables[table_name]
        response, column_count = update_databundle_entity_metadata(
            post,
            verification_info,
            firestore_path=str(live_table["path"]),
            description=str(table_metadata.get("description") or ""),
            fields=list(table_metadata.get("schema", {}).get("fields", [])),
        )
        rating = response.get("usabilityRating") or rating
        updated_tables += 1
        updated_columns += column_count
    return updated_tables, updated_columns, rating

def kaggle_databundle_files(session, headers: dict[str, str], basics: dict) -> dict[str, dict]:
    data = basics.get("data") or {}
    root_path = data.get("firestorePath")
    version_id = data.get("versionId")
    dataset_id = basics.get("datasetId")
    if not root_path or not version_id or not dataset_id:
        raise RuntimeError(f"Missing Kaggle databundle identity in dataset basics: {basics}")
    verification_info = {
        "databundleVersionId": version_id,
        "datasetId": dataset_id,
    }
    paths = [
        root_path,
        f"{root_path}/directories/exports/directories/csv",
        f"{root_path}/directories/exports/directories/parquet",
    ]
    files: dict[str, dict] = {}
    for firestore_path in paths:
        children = kaggle_internal_post(
            session,
            headers,
            "datasets.databundles.DatabundleService/GetDatabundleExternalChildren",
            {
                "verificationInfo": verification_info,
                "firestorePath": firestore_path,
                "offset": 0,
                "count": 200,
                "depth": 1,
                "enforceMaxDepthConstraint": False,
            },
        )
        for file_info in children.get("files") or []:
            relative_url = file_info.get("relativeUrl")
            if relative_url:
                files[str(relative_url)] = file_info
    return files

def update_kaggle_databundle_metadata_external(metadata: dict, basics: dict) -> None:
    session, headers = kaggle_internal_metadata_session()
    def post(route: str, body: dict) -> dict:
        return kaggle_internal_post(session, headers, route, body)

    data = basics.get("data") or {}
    verification_info = {
        "databundleVersionId": data.get("versionId"),
        "datasetId": basics.get("datasetId"),
    }
    files = kaggle_databundle_files(session, headers, basics)
    updated_files = 0
    updated_columns = 0
    updated_sqlite_tables = 0
    rating = {}
    for resource in metadata.get("resources") or []:
        resource_path = str(resource["path"])
        file_info = files.get(resource_path)
        if not file_info:
            raise RuntimeError(f"Kaggle live databundle file not found: {resource_path}")
        response, column_count = update_databundle_entity_metadata(
            post,
            verification_info,
            firestore_path=str(file_info["path"]),
            description=str(resource.get("description") or ""),
            fields=list(resource.get("schema", {}).get("fields", [])),
        )
        rating = response.get("usabilityRating") or {}
        updated_files += 1
        updated_columns += column_count
        if resource_path == "openoppsdb.sqlite":
            sqlite_deadline = time.time() + float(
                os.environ.get("OPENOPPS_KAGGLE_SQLITE_INDEX_WAIT_SECONDS", "1200")
            )
            while True:
                try:
                    table_count, table_column_count, table_rating = (
                        update_sqlite_table_metadata_external(
                            post,
                            verification_info,
                            file_info,
                        )
                    )
                    break
                except RuntimeError as exc:
                    if "Kaggle SQLite indexer did not index" not in str(exc):
                        raise
                    if time.time() >= sqlite_deadline:
                        raise
                    print(
                        "Waiting for Kaggle SQLite indexer metadata:",
                        json.dumps(
                            {
                                "path": resource_path,
                                "reason": str(exc),
                            },
                            sort_keys=True,
                        ),
                    )
                    time.sleep(30)
                    files = kaggle_databundle_files(session, headers, basics)
                    file_info = files.get(resource_path)
                    if not file_info:
                        raise RuntimeError(
                            f"Kaggle live databundle file not found: {resource_path}"
                        )
            updated_sqlite_tables += table_count
            updated_columns += table_column_count
            rating = table_rating or rating
    print(
        "Updated Kaggle live databundle metadata:",
        json.dumps(
            {
                "files": updated_files,
                "sqliteTables": updated_sqlite_tables,
                "columns": updated_columns,
                "usabilityScore": rating.get("score"),
                "columnDescriptionScore": rating.get("columnDescriptionScore"),
                "fileDescriptionScore": rating.get("fileDescriptionScore"),
            },
            sort_keys=True,
        ),
    )

require_kaggle_credentials()
install_openopps()
copy_latest_input_db()
download_dataset_assets()


In [ ]:
openopps_env = os.environ.copy()
openopps_env["OPENOPPS_DB_URL"] = f"sqlite:///{DB_PATH}"
openopps_env["OPENOPPS_CACHE_ENABLED"] = "false"
for key, value in OPENOPPS_SYNC_ENV_DEFAULTS.items():
    openopps_env.setdefault(key, value)

run(["openopps", "admin", "db", "init"], env=openopps_env)
print(f"OpenOpps bounded jobs sync timeout: {KAGGLE_SYNC_TIMEOUT_SECONDS:g}s")
print(f"OpenOpps bounded jobs sync route limit: {KAGGLE_JOB_ROUTE_LIMIT}")
sync_metrics = run_sync_metrics(
    OUTPUT_DIR / "sync_metrics.json",
    env=openopps_env,
    timeout_seconds=KAGGLE_SYNC_TIMEOUT_SECONDS,
)
skill_backfill = backfill_openopps_skill_tables(DB_PATH)
print("OpenOpps skill backfill:", json.dumps(skill_backfill, sort_keys=True))
status = run_json(
    ["openopps", "status", "--json"],
    OUTPUT_DIR / "status.json",
    env=openopps_env,
)
coverage = run_json(
    ["openopps", "providers", "coverage", "--json"],
    OUTPUT_DIR / "coverage.json",
    env=openopps_env,
)


In [ ]:
quality = write_public_bundle()
print("OpenOpps snapshot quality:", json.dumps({
    "status": quality["status"],
    "hardBlockers": quality["hardBlockers"],
    "warnings": quality["warnings"],
    "counts": {
        "jobs": quality["counts"].get("jobs"),
        "job_versions": quality["counts"].get("job_versions"),
        "job_version_skills": quality["counts"].get("job_version_skills"),
        "job_version_skill_keywords": quality["counts"].get("job_version_skill_keywords"),
        "openopps_tables": quality["counts"].get("openopps_tables"),
        "openopps_columns": quality["counts"].get("openopps_columns"),
    },
}, sort_keys=True))

for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name, path.stat().st_size)


In [ ]:
message = f"Scheduled OpenOpps active-job snapshot {datetime.now(UTC).isoformat()}"
require_kaggle_credentials()
metadata_session, metadata_headers = kaggle_internal_metadata_session()
previous_basics = kaggle_dataset_basics(metadata_session, metadata_headers)
previous_version = int(previous_basics.get("datasetVersionNumber") or 0)

run([
    "kaggle",
    "datasets",
    "version",
    "-p",
    str(OUTPUT_DIR),
    "-m",
    message,
    "-q",
    "-t",
    "-r",
    "zip",
])
published_basics = wait_for_new_live_dataset_version(previous_version)
update_kaggle_dataset_file_metadata(published_basics)
run(["kaggle", "datasets", "status", DATASET_ID, "--format", "json"])
run(["kaggle", "datasets", "files", DATASET_ID, "--page-size", "200"])
